
# Notebook 09 — Daily Attack Scheduling and Channel Sequencing

## Mubsir: AI-Based Detection of Channel-Rotating Low-and-Slow Insider Data Exfiltration

### Objective

This notebook transforms the 400 approved attacker-scenario assignments from Notebook 08 into feasible daily attack schedules.

The goal is to determine when each attacker operates, which exfiltration channels are used on each active day, and how channel rotation and synchronization occur throughout the attack window.

### Main Tasks

1. **Daily Activity Eligibility:** Identify eligible attack days and naturally available channels using the original daily behavioral records.
2. **Forced Off-Days:** Preserve all approved Forced Off-Days and strictly prohibit injection on those days.
3. **Channel Sequencing:** Construct chronological channel-use sequences for each attacker.
4. **Channel Rotation:** Schedule transitions between Email, USB, and HTTP according to the approved switching level.
5. **Channel Synchronization:** Evaluate and schedule feasible two-channel and three-channel activity on the same day.
6. **Stealth Compliance:** Preserve approved stealth levels and prepare for frequency-based injection planning.
7. **Validation:** Verify schedule feasibility, channel coverage, switching compliance, synchronization, and consistency with the approved assignments.

### Inputs

- Approved attacker-scenario assignments from Notebook 08.
- Original daily Email, USB, and HTTP event-count records.
- Approved scenario constraints and switching feasibility evidence.

### Core Constraints

- Preserve all 400 approved attacker assignments.
- Do not modify approved attack windows, durations, channel configurations, switching levels, or stealth levels.
- Never schedule attacks on Forced Off-Days.
- Schedule channels only on naturally eligible days.
- Use event counts and activity frequency only, excluding payload sizes and data volumes.
- Keep the historical Behavioral Baseline unchanged.
- Do not introduce artificial user activity or unsupported channel combinations.
- Distinguish channel rotation across days from channel synchronization within the same day.

### Expected Outputs

- Daily attack schedules for the 400 approved attackers.
- Chronological channel sequences.
- Channel rotation and synchronization reports.
- Per-user and per-day scheduling validation results.
- A validated daily scheduling handoff for subsequent attack injection.

### Execution Principle

Scheduling decisions will be derived from observed user activity and approved scenario constraints rather than imposed arbitrary patterns.

All schedules must pass the required feasibility and integrity checks before final export.



## 9.1 — Approved Inputs & Integrity Verification

### Objective

Load and verify the final attacker-scenario assignments approved in Notebook 08 before constructing any daily attack schedules.

This section establishes a trusted, read-only input for the Pre-Injection Dry Run.

### Primary Input

**Notebook 08 — Attacker Selection and Scenario Assignment**

`reports/08_Attacker_Selection_and_Scenario_Assignment/approved_attacker_scenarios_r62.parquet`

The approved file contains 400 unique attacker-scenario assignments, including:

- Attacker identity and approved scenario option.
- Attack window and duration.
- Channel configuration.
- Switching and stealth levels.
- Required Forced Off-Days and proposed active days.
- Approved feasibility evidence and source-validation status.

### Supporting Evidence

The following Notebook 08 artifacts are used to verify the approved handoff:

- `export_manifest.json`
- `assignment_distribution_qa.csv`
- `assignment_quality_qa.csv`
- `independent_source_qa.csv`

### Verification Procedure

1. Locate the New_Attack_Simulation project root.
2. Verify the presence of all required handoff files.
3. Verify the approved Parquet file against its recorded SHA-256 checksum.
4. Load the 400 approved assignments.
5. Check primary-key uniqueness and mandatory fields.
6. Verify approval and source-validation statuses.
7. Reconcile the approved duration, channel, switching, and stealth distributions.
8. Verify duration-based Forced Off-Day configurations and active-day totals.
9. Confirm that the supporting QA reports contain no failed checks.

### Constraints

- Treat all approved assignments as immutable.
- Do not change users, windows, channels, durations, switching, stealth, or Forced Off-Days.
- Do not recalculate the historical Baseline.
- Do not generate daily schedules or synchronization patterns.
- Do not inject events or modify source files.
- Do not export new files in this section.

### Expected Outcome

A verified in-memory DataFrame containing the 400 approved attacker assignments, ready for daily eligibility reconstruction and channel sequencing in the subsequent sections.


In [1]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.1
# APPROVED INPUTS & INTEGRITY VERIFICATION
#
# Read-only | No scheduling | No injection | No exports
# ============================================================

from pathlib import Path
import hashlib
import json

import pandas as pd
import pyarrow.parquet as pq

from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Locate project root
# ------------------------------------------------------------

cwd_09 = Path.cwd().resolve()

root_candidates_09 = [
    cwd_09,
    *cwd_09.parents,
    cwd_09 / "New_Attack_Simulation",
    cwd_09.parent / "New_Attack_Simulation",
]

ROOT_09 = next(
    (
        path for path in root_candidates_09
        if (
            path
            / "reports"
            / "08_Attacker_Selection_and_Scenario_Assignment"
            / "approved_attacker_scenarios_r62.parquet"
        ).is_file()
    ),
    None,
)

if ROOT_09 is None:
    raise FileNotFoundError(
        "Cannot locate the approved Notebook 08 handoff. "
        f"Current directory: {cwd_09}"
    )

REPORT_08_09 = (
    ROOT_09
    / "reports"
    / "08_Attacker_Selection_and_Scenario_Assignment"
)

REPORT_09 = (
    ROOT_09
    / "reports"
    / "09_PreInjection_Dry_Run"
)

print("STEP 1 — Project root located: PASS")
print("Project root:", ROOT_09)


# ------------------------------------------------------------
# STEP 2 — Define required input paths
# ------------------------------------------------------------

INPUT_PATHS_09 = {
    "Approved Assignments": (
        REPORT_08_09
        / "approved_attacker_scenarios_r62.parquet"
    ),
    "Export Manifest": (
        REPORT_08_09 / "export_manifest.json"
    ),
    "Assignment Distribution QA": (
        REPORT_08_09 / "assignment_distribution_qa.csv"
    ),
    "Assignment Quality QA": (
        REPORT_08_09 / "assignment_quality_qa.csv"
    ),
    "Independent Source QA": (
        REPORT_08_09 / "independent_source_qa.csv"
    ),
}

missing_files_091 = [
    str(path)
    for path in INPUT_PATHS_09.values()
    if not path.is_file()
]

if missing_files_091:
    raise FileNotFoundError(
        "Required input files are missing:\n"
        + "\n".join(missing_files_091)
    )

print("STEP 2 — Required handoff files located: PASS")


# ------------------------------------------------------------
# STEP 3 — Verify SHA-256 integrity
# ------------------------------------------------------------

with open(
    INPUT_PATHS_09["Export Manifest"],
    "r",
    encoding="utf-8",
) as handle:
    manifest_09 = json.load(handle)


def sha256_file_091(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


assignment_filename_091 = (
    INPUT_PATHS_09["Approved Assignments"].name
)

manifest_record_091 = (
    manifest_09.get("files", {})
    .get(assignment_filename_091)
)

if not manifest_record_091:
    raise RuntimeError(
        "Approved assignment checksum missing "
        "from export manifest."
    )

expected_hash_091 = manifest_record_091["sha256"]

actual_hash_091 = sha256_file_091(
    INPUT_PATHS_09["Approved Assignments"]
)

if actual_hash_091 != expected_hash_091:
    raise RuntimeError(
        "Approved assignment checksum mismatch. "
        "Handoff integrity cannot be confirmed."
    )

assert manifest_09["assignment_status"] == "APPROVED"
assert manifest_09["source_validation_status"] == "SOURCE_QA_PASS"
assert manifest_09["attackers"] == 400

print("STEP 3 — Manifest and SHA-256 integrity: PASS")


# ------------------------------------------------------------
# STEP 4 — Load approved assignments
# ------------------------------------------------------------

assignments_09 = pd.read_parquet(
    INPUT_PATHS_09["Approved Assignments"]
)

required_columns_091 = [
    "user",
    "scenario_option_id",
    "window_workdays",
    "window_start",
    "window_end",
    "channel_configuration",
    "switching_level",
    "stealth_level",
    "stealth_multiplier",
    "forced_offdays",
    "proposed_active_days",
    "set_switch_bitmask",
    "source_validation_status",
    "assignment_status",
]

missing_columns_091 = [
    col for col in required_columns_091
    if col not in assignments_09.columns
]

if missing_columns_091:
    raise RuntimeError(
        "Required assignment columns missing: "
        + ", ".join(missing_columns_091)
    )

assignments_09["window_start"] = pd.to_datetime(
    assignments_09["window_start"]
).dt.normalize()

assignments_09["window_end"] = pd.to_datetime(
    assignments_09["window_end"]
).dt.normalize()

assert len(assignments_09) == 400
assert assignments_09["user"].is_unique
assert assignments_09["scenario_option_id"].is_unique

assert assignments_09[
    required_columns_091
].notna().all().all()

assert assignments_09[
    "assignment_status"
].eq("APPROVED").all()

assert assignments_09[
    "source_validation_status"
].eq("SOURCE_QA_PASS").all()

print("STEP 4 — Approved assignments loaded: PASS")
print("Assignments:", len(assignments_09))
print("Columns:", len(assignments_09.columns))


# ------------------------------------------------------------
# STEP 5 — Verify exact approved distributions
# ------------------------------------------------------------

CHANNEL_TARGETS_09 = {
    "E+U": 117,
    "E+H": 117,
    "U+H": 27,
    "E+U+H": 139,
}

DURATION_TARGETS_09 = {
    5: 50,
    10: 70,
    20: 70,
    30: 70,
    40: 70,
    50: 70,
}

SWITCHING_TARGETS_09 = {
    "Low": 133,
    "Medium": 134,
    "High": 133,
}

STEALTH_TARGETS_09 = {
    "Low": 133,
    "Medium": 133,
    "High": 134,
}

targets_091 = {
    "channel_configuration": CHANNEL_TARGETS_09,
    "window_workdays": DURATION_TARGETS_09,
    "switching_level": SWITCHING_TARGETS_09,
    "stealth_level": STEALTH_TARGETS_09,
}

distribution_rows_091 = []

for column, targets in targets_091.items():

    actual_counts = (
        assignments_09[column]
        .value_counts()
    )

    for category, expected in targets.items():

        actual = int(
            actual_counts.get(category, 0)
        )

        distribution_rows_091.append({
            "dimension": column,
            "category": category,
            "expected": expected,
            "actual": actual,
            "status": (
                "PASS" if actual == expected else "FAIL"
            ),
        })

distribution_qa_09 = pd.DataFrame(
    distribution_rows_091
)

if not distribution_qa_09["status"].eq("PASS").all():
    display(distribution_qa_09)
    raise AssertionError(
        "Approved assignment distribution mismatch."
    )

print("STEP 5 — Exact assignment distributions: PASS")


# ------------------------------------------------------------
# STEP 6 — Verify duration-specific Forced Off-Days
# ------------------------------------------------------------

ALLOWED_OFFDAYS_09 = {
    5: [0],
    10: [1, 2],
    20: [2, 3, 4],
    30: [4, 5, 6],
    40: [6, 7, 8],
    50: [8, 9, 10],
}

for duration, allowed in ALLOWED_OFFDAYS_09.items():

    subset = assignments_09.loc[
        assignments_09["window_workdays"].eq(duration)
    ]

    assert subset["forced_offdays"].isin(allowed).all()

assert (
    assignments_09["forced_offdays"]
    + assignments_09["proposed_active_days"]
).eq(
    assignments_09["window_workdays"]
).all()

total_forced_offdays_09 = int(
    assignments_09["forced_offdays"].sum()
)

assert total_forced_offdays_09 == int(
    manifest_09["forced_offdays"]
)

print("STEP 6 — Forced Off-Day integrity: PASS")
print("Total Forced Off-Days:", total_forced_offdays_09)


# ------------------------------------------------------------
# STEP 7 — Verify supporting QA reports
# ------------------------------------------------------------

quality_qa_09 = pd.read_csv(
    INPUT_PATHS_09["Assignment Quality QA"]
)

source_qa_09 = pd.read_csv(
    INPUT_PATHS_09["Independent Source QA"]
)

distribution_source_09 = pd.read_csv(
    INPUT_PATHS_09["Assignment Distribution QA"]
)

for name, report in {
    "Assignment Quality": quality_qa_09,
    "Independent Source": source_qa_09,
    "Assignment Distribution": distribution_source_09,
}.items():

    if "status" not in report.columns:
        raise RuntimeError(
            f"{name} report is missing the status column."
        )

    if report.empty or not report["status"].eq("PASS").all():
        raise RuntimeError(
            f"{name} report contains failed checks."
        )

assert len(source_qa_09) == int(
    manifest_09["source_qa_checks"]
)

assert len(quality_qa_09) == int(
    manifest_09["quality_qa_checks"]
)

# Check that QA files still match recorded export hashes.
for input_name in [
    "Assignment Quality QA",
    "Independent Source QA",
    "Assignment Distribution QA",
]:

    path = INPUT_PATHS_09[input_name]
    recorded = manifest_09["files"].get(path.name)

    if not recorded:
        raise RuntimeError(
            f"Missing manifest record: {path.name}"
        )

    if sha256_file_091(path) != recorded["sha256"]:
        raise RuntimeError(
            f"QA file checksum mismatch: {path.name}"
        )

print("STEP 7 — Supporting QA reports: PASS")


# ------------------------------------------------------------
# STEP 8 — Confirm scheduling has not started
# ------------------------------------------------------------

# Copy intended for downstream read-only analysis.
# No daily attack schedule is created in this section.

approved_assignments_09 = (
    assignments_09
    .sort_values(["user", "scenario_option_id"])
    .reset_index(drop=True)
    .copy(deep=True)
)

print("STEP 8 — Approved handoff ready: PASS")


# ------------------------------------------------------------
# STEP 9 — Display input overview
# ------------------------------------------------------------

print("\nAPPROVED CHANNEL DISTRIBUTION")
display(
    approved_assignments_09[
        "channel_configuration"
    ].value_counts().rename_axis(
        "configuration"
    ).reset_index(name="attackers")
)

print("\nAPPROVED DURATION DISTRIBUTION")
display(
    approved_assignments_09[
        "window_workdays"
    ].value_counts().sort_index().rename_axis(
        "duration"
    ).reset_index(name="attackers")
)

print("\nINPUT SCHEMA")
display(pd.DataFrame([
    {
        "column": column,
        "dtype": str(dtype),
    }
    for column, dtype in approved_assignments_09.dtypes.items()
]))


# ------------------------------------------------------------
# STEP 10 — Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NOTEBOOK 09 — SECTION 9.1 SUMMARY")
print("=" * 65)

print("Approved attackers       :", len(approved_assignments_09))
print("Unique users             :", approved_assignments_09["user"].nunique())
print("Approved scenario IDs    :", approved_assignments_09["scenario_option_id"].nunique())
print("Forced Off-Days          :", total_forced_offdays_09)
print("Assignment distribution  : PASS")
print("Source validation QA     :", len(source_qa_09), "PASS")
print("Assignment quality QA    :", len(quality_qa_09), "PASS")
print("SHA-256 integrity        : PASS")
print("Daily schedules generated: NO")
print("Channel synchronization  : NOT PERFORMED")
print("Original files modified  : NO")
print("Files exported           : NO")
print("Future report directory  :", REPORT_09)
print("=" * 65)

print("SECTION 9.1 — PASS")


STEP 1 — Project root located: PASS
Project root: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation
STEP 2 — Required handoff files located: PASS
STEP 3 — Manifest and SHA-256 integrity: PASS
STEP 4 — Approved assignments loaded: PASS
Assignments: 400
Columns: 23
STEP 5 — Exact assignment distributions: PASS
STEP 6 — Forced Off-Day integrity: PASS
Total Forced Off-Days: 1670
STEP 7 — Supporting QA reports: PASS
STEP 8 — Approved handoff ready: PASS

APPROVED CHANNEL DISTRIBUTION


,configuration,attackers
0,E+U+H,139
1,E+H,117
2,E+U,117
3,U+H,27



APPROVED DURATION DISTRIBUTION


,duration,attackers
0,5,50
1,10,70
2,20,70
3,30,70
4,40,70
5,50,70



INPUT SCHEMA


,column,dtype
0,user,str
1,scenario_option_id,uint32
2,max_abs_recurrence_gap_pp,float64
3,max_positive_recurrence_increase_pp,float64
4,all_selected_channels_have_reference,bool
5,window_workdays,int64
6,window_start,datetime64[ms]
7,window_end,datetime64[ms]
8,channel_configuration,category
9,channel_count,int64



NOTEBOOK 09 — SECTION 9.1 SUMMARY
Approved attackers       : 400
Unique users             : 400
Approved scenario IDs    : 400
Forced Off-Days          : 1670
Assignment distribution  : PASS
Source validation QA     : 15 PASS
Assignment quality QA    : 13 PASS
SHA-256 integrity        : PASS
Daily schedules generated: NO
Channel synchronization  : NOT PERFORMED
Original files modified  : NO
Files exported           : NO
Future report directory  : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\09_PreInjection_Dry_Run
SECTION 9.1 — PASS



## 9.2 — Approved Daily Window Reconstruction & Channel Availability

### Objective

Construct the day-level timeline for the 400 approved attacker scenarios using the original User-Day dataset and the previously approved workday eligibility rule.

This section identifies the exact days and naturally available channels needed for subsequent attack scheduling, without revisiting earlier user eligibility, scenario selection, or feasibility decisions.

### Inputs

**Notebook 08 — Approved Attacker Assignments**

- `approved_assignments_09` — The 400 immutable attacker-scenario assignments.

**Data Engineering — Original User-Day Records**

- `user_day_r62_v1.parquet`

Required fields:

| Field | Purpose |
|---|---|
| `user` | User identifier |
| `day` | Original activity date |
| `is_weekend` | Existing calendar eligibility indicator |
| `logon_count` | Existing workday eligibility evidence |
| `external_email_count` | Natural Email activity frequency |
| `usb_copy_count` | Natural USB copy frequency |
| `www_upload_count` | Natural HTTP upload frequency |

### Methodology

#### 1. Reuse Approved Workday Eligibility

Apply the existing eligibility rule established in Notebook 05:

- `is_weekend == 0`
- `logon_count > 0`

This rule is reused without recalibration or modification.

#### 2. Reconstruct Approved Attack Windows

Extract eligible user-day records that fall within each attacker's approved `window_start` and `window_end`.

Preserve the original day order and generate a sequential workday index for scheduling.

#### 3. Locate Approved Forced Off-Days

Identify naturally inactive days using zero original activity across Email, USB, and HTTP.

These days correspond to previously approved Forced Off-Days and are marked as strictly prohibited for injection.

No additional Off-Days are introduced.

#### 4. Identify Natural Channel Availability

For each eligible day, identify which channels have positive original event counts.

Separate:

- Natural activity across all three channels.
- Availability restricted to the attacker's approved channel configuration.

#### 5. Identify Potential Synchronization Opportunities

Record days where two or three approved channels are naturally available simultaneously.

These are descriptive opportunities only. No synchronization or daily channel assignments are performed in this section.

#### 6. Timeline Integrity

Verify that the reconstructed daily timeline respects the immutable approved assignments:

- All 400 attackers are represented.
- Each attack window contains its approved eligible workday count.
- Window boundaries match the approved dates.
- Daily Forced Off-Days and active-day counts agree with the approved assignment.
- Original event counts remain unchanged.

### Outputs

In-memory objects:

- `source_user_day_09` — Original daily records for the selected population.
- `daily_window_09` — Reconstructed eligible daily attack-window timeline.
- `daily_window_summary_09` — Per-attacker daily availability summary.
- `daily_window_qa_09` — Timeline integrity checks.
- `channel_opportunity_summary_09` — Descriptive natural channel availability.

### Constraints

- Preserve the approved 400 attacker assignments.
- Do not redefine workday eligibility.
- Do not recalculate historical Baselines.
- Do not change Forced Off-Day counts or scenario windows.
- Do not recompute approved switching feasibility.
- Do not assign primary channels, switching sequences, or synchronization days.
- Use event counts and activity frequency only.
- No injection, source modification, or exports.

### Expected Outcome

A clean, validated user-day timeline containing the approved Forced Off-Day positions and naturally available channels for each attacker.

This timeline serves as the input for synchronization-opportunity analysis and subsequent channel-sequence construction.


In [2]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.2
# APPROVED DAILY WINDOW RECONSTRUCTION
# & CHANNEL AVAILABILITY
#
# Read-only | No scheduling | No injection | No exports
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Locate original User-Day source
# ------------------------------------------------------------

SOURCE_PATH_092 = (
    ROOT_09.parent
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
    / "processed"
    / "user_day_r62_v1.parquet"
)

if not SOURCE_PATH_092.is_file():
    raise FileNotFoundError(
        "Original User-Day source not found:\n"
        f"{SOURCE_PATH_092}"
    )

COUNT_COLS_092 = {
    "E": "external_email_count",
    "U": "usb_copy_count",
    "H": "www_upload_count",
}

REQUIRED_SOURCE_COLS_092 = [
    "user",
    "day",
    "is_weekend",
    "logon_count",
    *COUNT_COLS_092.values(),
]

print("STEP 1 — Original User-Day source located: PASS")
print("Source:", SOURCE_PATH_092)


# ------------------------------------------------------------
# STEP 2 — Load only required User-Day fields
# ------------------------------------------------------------

source_user_day_09 = pd.read_parquet(
    SOURCE_PATH_092,
    columns=REQUIRED_SOURCE_COLS_092,
)

source_user_day_09["user"] = (
    source_user_day_09["user"].astype("string")
)

source_user_day_09["day"] = pd.to_datetime(
    source_user_day_09["day"]
).dt.normalize()

approved_092 = (
    approved_assignments_09
    .copy(deep=True)
)

approved_092["user"] = (
    approved_092["user"].astype("string")
)

for col in ["window_start", "window_end"]:
    approved_092[col] = pd.to_datetime(
        approved_092[col]
    ).dt.normalize()

# Retain only records belonging to approved attackers.
source_user_day_09 = source_user_day_09.loc[
    source_user_day_09["user"].isin(
        approved_092["user"]
    )
].copy()

assert source_user_day_09["user"].nunique() == 400

assert not source_user_day_09.duplicated(
    ["user", "day"]
).any()

assert source_user_day_09[
    REQUIRED_SOURCE_COLS_092
].notna().all().all()

assert source_user_day_09[
    list(COUNT_COLS_092.values())
].ge(0).all().all()

print("STEP 2 — Selected User-Day records loaded: PASS")
print("Selected users:", source_user_day_09["user"].nunique())


# ------------------------------------------------------------
# STEP 3 — Reuse previously approved workday eligibility
#
# This is the established Notebook 05 rule,
# NOT a new eligibility decision.
# ------------------------------------------------------------

eligible_092 = source_user_day_09.loc[
    source_user_day_09["is_weekend"].eq(0)
    &
    source_user_day_09["logon_count"].gt(0)
].copy()

print("STEP 3 — Approved workday rule applied: PASS")


# ------------------------------------------------------------
# STEP 4 — Reconstruct approved daily windows
# ------------------------------------------------------------

ASSIGNMENT_COLS_092 = [
    "user",
    "scenario_option_id",
    "window_workdays",
    "window_start",
    "window_end",
    "channel_configuration",
    "switching_level",
    "stealth_level",
    "stealth_multiplier",
    "forced_offdays",
    "proposed_active_days",
    "set_switch_bitmask",
]

daily_window_09 = eligible_092.merge(
    approved_092[ASSIGNMENT_COLS_092],
    on="user",
    how="inner",
    validate="many_to_one",
)

daily_window_09 = daily_window_09.loc[
    daily_window_09["day"].ge(
        daily_window_09["window_start"]
    )
    &
    daily_window_09["day"].le(
        daily_window_09["window_end"]
    )
].copy()

daily_window_09 = (
    daily_window_09
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

# Chronological eligible-workday position.
daily_window_09["workday_index"] = (
    daily_window_09
    .groupby("user")
    .cumcount()
    .add(1)
)

assert not daily_window_09.duplicated(
    ["user", "day"]
).any()

print("STEP 4 — Approved daily windows reconstructed: PASS")


# ------------------------------------------------------------
# STEP 5 — Map original channel activity
#
# Positive original count = naturally active.
# This does not represent injected activity.
# ------------------------------------------------------------

for channel, count_col in COUNT_COLS_092.items():

    daily_window_09[f"natural_{channel}"] = (
        daily_window_09[count_col].gt(0)
    )

CHANNEL_BITS_092 = {
    "E": 1,
    "U": 2,
    "H": 4,
}

MASK_TO_CHANNELS_092 = {
    0: "NONE",
    1: "E",
    2: "U",
    3: "E+U",
    4: "H",
    5: "E+H",
    6: "U+H",
    7: "E+U+H",
}

daily_window_09["natural_channel_mask"] = (
    daily_window_09["natural_E"].astype(int) * 1
    +
    daily_window_09["natural_U"].astype(int) * 2
    +
    daily_window_09["natural_H"].astype(int) * 4
)

daily_window_09["natural_channel_set"] = (
    daily_window_09["natural_channel_mask"]
    .map(MASK_TO_CHANNELS_092)
)

daily_window_09["natural_channel_count"] = (
    daily_window_09[
        ["natural_E", "natural_U", "natural_H"]
    ]
    .sum(axis=1)
    .astype(int)
)

print("STEP 5 — Natural channel activity mapped: PASS")


# ------------------------------------------------------------
# STEP 6 — Locate approved Forced Off-Day positions
#
# Forced Off-Days are derived from the existing
# natural daily evidence, not chosen or created here.
# ------------------------------------------------------------

daily_window_09["is_forced_offday"] = (
    daily_window_09["natural_channel_mask"].eq(0)
)

daily_window_09["is_active_day"] = (
    ~daily_window_09["is_forced_offday"]
)

daily_window_09["day_status"] = np.where(
    daily_window_09["is_forced_offday"],
    "FORCED_OFFDAY",
    "ACTIVE",
)

# Explicit downstream prohibition marker.
daily_window_09["injection_allowed"] = (
    ~daily_window_09["is_forced_offday"]
)

print("STEP 6 — Approved Forced Off-Day positions located: PASS")


# ------------------------------------------------------------
# STEP 7 — Restrict availability to approved channels
# ------------------------------------------------------------

CONFIG_MASKS_092 = {
    "E+U": 3,
    "E+H": 5,
    "U+H": 6,
    "E+U+H": 7,
}

daily_window_09["approved_channel_mask"] = (
    daily_window_09["channel_configuration"]
    .map(CONFIG_MASKS_092)
)

assert daily_window_09[
    "approved_channel_mask"
].notna().all()

daily_window_09["available_channel_mask"] = (
    daily_window_09["natural_channel_mask"]
    .to_numpy(dtype=np.int64)
    &
    daily_window_09["approved_channel_mask"]
    .to_numpy(dtype=np.int64)
)

daily_window_09["available_channel_set"] = (
    daily_window_09["available_channel_mask"]
    .map(MASK_TO_CHANNELS_092)
)

MASK_COUNTS_092 = {
    mask: bin(mask).count("1")
    for mask in range(8)
}

daily_window_09["available_channel_count"] = (
    daily_window_09["available_channel_mask"]
    .map(MASK_COUNTS_092)
    .astype(int)
)

daily_window_09["has_approved_channel"] = (
    daily_window_09["available_channel_count"].gt(0)
)

# An overall active day may still lack activity
# in the attacker's approved channel set.
daily_window_09["active_without_approved_channel"] = (
    daily_window_09["is_active_day"]
    &
    ~daily_window_09["has_approved_channel"]
)

print("STEP 7 — Approved-channel availability mapped: PASS")


# ------------------------------------------------------------
# STEP 8 — Identify potential same-day opportunities
#
# Descriptive evidence only.
# No synchronization decisions are made here.
# ------------------------------------------------------------

daily_window_09["potential_two_channel_day"] = (
    daily_window_09["available_channel_count"].ge(2)
)

daily_window_09["potential_three_channel_day"] = (
    daily_window_09["available_channel_count"].eq(3)
)

daily_window_09["potential_exactly_two_day"] = (
    daily_window_09["available_channel_count"].eq(2)
)

assert not daily_window_09.loc[
    daily_window_09["is_forced_offday"],
    "has_approved_channel",
].any()

assert not daily_window_09.loc[
    daily_window_09["is_forced_offday"],
    "potential_two_channel_day",
].any()

print("STEP 8 — Natural synchronization opportunities mapped: PASS")


# ------------------------------------------------------------
# STEP 9 — Minimal timeline integrity check
#
# Notebook 08 already validated source consistency.
# Here we only ensure the constructed daily timeline
# respects the approved immutable assignment.
# ------------------------------------------------------------

actual_daily_092 = (
    daily_window_09
    .groupby("user")
    .agg(
        observed_workdays=("day", "size"),
        first_day=("day", "min"),
        last_day=("day", "max"),
        observed_forced_offdays=(
            "is_forced_offday", "sum"
        ),
        observed_active_days=(
            "is_active_day", "sum"
        ),
        available_channel_days=(
            "has_approved_channel", "sum"
        ),
        potential_two_channel_days=(
            "potential_two_channel_day", "sum"
        ),
        potential_three_channel_days=(
            "potential_three_channel_day", "sum"
        ),
        active_without_approved_days=(
            "active_without_approved_channel", "sum"
        ),
    )
    .reset_index()
)

daily_window_summary_09 = approved_092[
    [
        "user",
        "scenario_option_id",
        "window_workdays",
        "window_start",
        "window_end",
        "channel_configuration",
        "forced_offdays",
        "proposed_active_days",
    ]
].merge(
    actual_daily_092,
    on="user",
    how="left",
    validate="one_to_one",
)

daily_window_summary_09["workdays_match"] = (
    daily_window_summary_09["observed_workdays"]
    .eq(daily_window_summary_09["window_workdays"])
)

daily_window_summary_09["start_match"] = (
    daily_window_summary_09["first_day"]
    .eq(daily_window_summary_09["window_start"])
)

daily_window_summary_09["end_match"] = (
    daily_window_summary_09["last_day"]
    .eq(daily_window_summary_09["window_end"])
)

daily_window_summary_09["offdays_match"] = (
    daily_window_summary_09["observed_forced_offdays"]
    .eq(daily_window_summary_09["forced_offdays"])
)

daily_window_summary_09["active_days_match"] = (
    daily_window_summary_09["observed_active_days"]
    .eq(daily_window_summary_09["proposed_active_days"])
)

INTEGRITY_COLS_092 = [
    "workdays_match",
    "start_match",
    "end_match",
    "offdays_match",
    "active_days_match",
]

daily_window_summary_09["timeline_valid"] = (
    daily_window_summary_09[
        INTEGRITY_COLS_092
    ].all(axis=1)
)

daily_window_qa_09 = pd.DataFrame([
    {
        "check": "400 approved daily windows",
        "status": (
            "PASS"
            if len(daily_window_summary_09) == 400
            else "FAIL"
        ),
    },
    {
        "check": "Unique user-day timeline keys",
        "status": (
            "PASS"
            if not daily_window_09.duplicated(
                ["user", "day"]
            ).any()
            else "FAIL"
        ),
    },
    {
        "check": "Approved window boundaries and counts",
        "status": (
            "PASS"
            if daily_window_summary_09[
                INTEGRITY_COLS_092
            ].all().all()
            else "FAIL"
        ),
    },
    {
        "check": "Approved total Forced Off-Days preserved",
        "status": (
            "PASS"
            if int(
                daily_window_09["is_forced_offday"].sum()
            ) == total_forced_offdays_09
            else "FAIL"
        ),
    },
    {
        "check": "No channel available on Forced Off-Days",
        "status": (
            "PASS"
            if not daily_window_09.loc[
                daily_window_09["is_forced_offday"],
                "has_approved_channel",
            ].any()
            else "FAIL"
        ),
    },
])

print("\nDAILY TIMELINE INTEGRITY")
display(daily_window_qa_09)

if not daily_window_qa_09["status"].eq("PASS").all():

    print("\nINVALID TIMELINES")
    display(
        daily_window_summary_09.loc[
            ~daily_window_summary_09["timeline_valid"]
        ].head(20)
    )

    raise RuntimeError(
        "Daily timeline reconstruction failed. "
        "Scheduling is blocked."
    )

print("STEP 9 — Daily timeline integrity: PASS")


# ------------------------------------------------------------
# STEP 10 — Summarize channel availability
# ------------------------------------------------------------

channel_opportunity_summary_09 = (
    daily_window_09
    .groupby("channel_configuration")
    .agg(
        attackers=("user", "nunique"),
        window_days=("day", "size"),
        forced_offdays=("is_forced_offday", "sum"),
        active_days=("is_active_day", "sum"),
        available_channel_days=(
            "has_approved_channel", "sum"
        ),
        active_without_approved_days=(
            "active_without_approved_channel", "sum"
        ),
        potential_two_channel_days=(
            "potential_two_channel_day", "sum"
        ),
        potential_three_channel_days=(
            "potential_three_channel_day", "sum"
        ),
    )
    .reindex(CHANNEL_TARGETS_09)
    .reset_index()
)

total_window_days_092 = len(daily_window_09)

total_active_days_092 = int(
    daily_window_09["is_active_day"].sum()
)

total_forced_days_092 = int(
    daily_window_09["is_forced_offday"].sum()
)

print("\nCHANNEL AVAILABILITY BY ASSIGNMENT")
display(channel_opportunity_summary_09)

print("\nDAILY WINDOW SAMPLE")
display(
    daily_window_09[
        [
            "user",
            "day",
            "workday_index",
            "day_status",
            "channel_configuration",
            "natural_channel_set",
            "available_channel_set",
            "external_email_count",
            "usb_copy_count",
            "www_upload_count",
        ]
    ].head(15)
)

print("STEP 10 — Daily availability summarized: PASS")


# ------------------------------------------------------------
# STEP 11 — Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NOTEBOOK 09 — SECTION 9.2 SUMMARY")
print("=" * 65)

print("Approved attackers        :", 400)
print("Reconstructed window-days :", total_window_days_092)
print("Active days               :", total_active_days_092)
print("Forced Off-Days           :", total_forced_days_092)

print(
    "Days with approved channels:",
    int(daily_window_09["has_approved_channel"].sum())
)

print(
    "Active without approved channel:",
    int(
        daily_window_09[
            "active_without_approved_channel"
        ].sum()
    )
)

print(
    "Potential 2+ channel days:",
    int(
        daily_window_09[
            "potential_two_channel_day"
        ].sum()
    )
)

print(
    "Potential 3-channel days :",
    int(
        daily_window_09[
            "potential_three_channel_day"
        ].sum()
    )
)

print(
    "Timeline QA              :",
    f"{len(daily_window_qa_09)}/"
    f"{len(daily_window_qa_09)} PASS"
)

print("Channel sequence scheduled: NO")
print("Synchronization scheduled : NO")
print("Original files modified   : NO")
print("Files exported            : NO")
print("=" * 65)

print("SECTION 9.2 — PASS")


STEP 1 — Original User-Day source located: PASS
Source: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\processed\user_day_r62_v1.parquet
STEP 2 — Selected User-Day records loaded: PASS
Selected users: 400
STEP 3 — Approved workday rule applied: PASS
STEP 4 — Approved daily windows reconstructed: PASS
STEP 5 — Natural channel activity mapped: PASS
STEP 6 — Approved Forced Off-Day positions located: PASS
STEP 7 — Approved-channel availability mapped: PASS
STEP 8 — Natural synchronization opportunities mapped: PASS

DAILY TIMELINE INTEGRITY


,check,status
0,400 approved daily windows,PASS
1,Unique user-day timeline keys,PASS
2,Approved window boundaries and counts,PASS
3,Approved total Forced Off-Days preserved,PASS
4,No channel available on Forced Off-Days,PASS


STEP 9 — Daily timeline integrity: PASS

CHANNEL AVAILABILITY BY ASSIGNMENT


,channel_configuration,attackers,window_days,forced_offdays,active_days,available_channel_days,active_without_approved_days,potential_two_channel_days,potential_three_channel_days
0,E+U,117,3480,531,2949,2949,0,963,0
1,E+H,117,3225,522,2703,2703,0,633,0
2,U+H,27,245,22,223,223,0,50,0
3,E+U+H,139,3800,595,3205,3205,0,1162,173



DAILY WINDOW SAMPLE


,user,day,workday_index,day_status,channel_configuration,natural_channel_set,available_channel_set,external_email_count,usb_copy_count,www_upload_count
0,AAF0819,2011-01-25,1,ACTIVE,E+U,E+U,E+U,2.0,3.0,0.0
1,AAF0819,2011-01-26,2,ACTIVE,E+U,U,U,0.0,2.0,0.0
2,AAF0819,2011-01-27,3,ACTIVE,E+U,E+U,E+U,3.0,1.0,0.0
3,AAF0819,2011-01-28,4,ACTIVE,E+U,U,U,0.0,1.0,0.0
4,AAF0819,2011-01-31,5,ACTIVE,E+U,E,E,4.0,0.0,0.0
5,ABD3426,2011-01-06,1,ACTIVE,U+H,E+U+H,U+H,2.0,7.0,1.0
6,ABD3426,2011-01-07,2,ACTIVE,U+H,E+U,U,7.0,9.0,0.0
7,ABD3426,2011-01-10,3,ACTIVE,U+H,E+U,U,5.0,4.0,0.0
8,ABD3426,2011-01-11,4,ACTIVE,U+H,E+U,U,4.0,9.0,0.0
9,ABD3426,2011-01-12,5,ACTIVE,U+H,E+U,U,3.0,5.0,0.0


STEP 10 — Daily availability summarized: PASS

NOTEBOOK 09 — SECTION 9.2 SUMMARY
Approved attackers        : 400
Reconstructed window-days : 10750
Active days               : 9080
Forced Off-Days           : 1670
Days with approved channels: 9080
Active without approved channel: 0
Potential 2+ channel days: 2808
Potential 3-channel days : 173
Timeline QA              : 5/5 PASS
Channel sequence scheduled: NO
Synchronization scheduled : NO
Original files modified   : NO
Files exported            : NO
SECTION 9.2 — PASS



## 9.3 — Synchronization Opportunity Analysis

### Objective

Analyze the natural same-day synchronization opportunities available to the 400 approved attackers, using the daily channel-availability evidence reconstructed in Section 9.2.

The purpose is to understand how synchronization opportunities are distributed across users, channel configurations, attack durations, and switching levels before introducing scheduling decisions.

### Inputs

**Approved daily-window evidence from Section 9.2**

- `daily_window_09` — Eligible user-day timeline and natural channel availability.
- `daily_window_summary_09` — Per-attacker window summary.
- `approved_assignments_09` — Immutable scenario assignments.

No new source files are required.

### Opportunity Definitions

**Two-Channel Opportunity**

An eligible active day where at least two channels from the attacker's approved configuration are naturally active.

**Exactly Two-Channel Opportunity**

An eligible active day where exactly two approved channels are naturally active.

**Three-Channel Opportunity**

An eligible active day where Email, USB, and HTTP are all naturally active and the attacker is assigned the E+U+H configuration.

**Synchronization-Capable Attacker**

An attacker with at least one naturally supported two-channel opportunity.

**Three-Channel-Capable Attacker**

An E+U+H attacker with at least one naturally supported three-channel opportunity.

These terms describe observed availability, not completed or guaranteed attack schedules.

### Analysis Procedure

1. Aggregate synchronization opportunities for each approved attacker.
2. Count users with and without two-channel opportunities.
3. Identify how many of the 139 E+U+H attackers have three-channel opportunities.
4. Distinguish exactly-two-channel opportunities from three-channel opportunities.
5. Examine opportunity distribution by channel configuration.
6. Compare opportunity availability across approved attack durations and switching levels.
7. Measure how concentrated synchronization opportunities are among users.
8. Identify naturally supported channel-pair combinations.

### Methodological Boundaries

- Do not assign synchronization days.
- Do not specify a minimum synchronization quota.
- Do not impose equal synchronization across channel configurations.
- Do not modify approved switching or stealth levels.
- Do not reconstruct previous scenario feasibility decisions.
- Do not recalculate the Behavioral Baseline.
- Do not modify Forced Off-Days or approved attack windows.
- Use natural channel activity indicators derived from event counts only.

### Validation

Verify that:

- All 400 approved attackers are represented exactly once.
- Per-user opportunity counts reconcile with Section 9.2.
- Three-channel opportunities occur only within E+U+H assignments.
- No synchronization opportunity occurs on a Forced Off-Day.
- Exactly-two and three-channel opportunities partition the two-or-more-channel opportunities.
- Opportunity statistics do not imply a completed daily schedule.

### Expected Outcome

A descriptive synchronization-opportunity profile for all 400 attackers, including per-user counts, channel-pair opportunities, three-channel availability, concentration statistics, and cross-dimensional summaries.

The resulting evidence will support joint switching and synchronization feasibility analysis in Section 9.4.


In [3]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.3
# SYNCHRONIZATION OPPORTUNITY ANALYSIS
#
# Descriptive analysis only
# No scheduling | No reassignment | No exports
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Reuse approved daily evidence
# ------------------------------------------------------------

daily_093 = daily_window_09.copy(deep=True)

assignments_093 = (
    approved_assignments_09
    .copy(deep=True)
)

assert len(assignments_093) == 400
assert assignments_093["user"].is_unique

assert daily_window_qa_09["status"].eq("PASS").all()

required_cols_093 = [
    "user",
    "day",
    "channel_configuration",
    "window_workdays",
    "switching_level",
    "is_forced_offday",
    "is_active_day",
    "available_channel_mask",
    "available_channel_count",
    "potential_two_channel_day",
    "potential_three_channel_day",
]

missing_cols_093 = [
    col for col in required_cols_093
    if col not in daily_093.columns
]

if missing_cols_093:
    raise RuntimeError(
        "Missing Section 9.2 fields: "
        + ", ".join(missing_cols_093)
    )

assert not daily_093.duplicated(
    ["user", "day"]
).any()

print("STEP 1 — Approved daily evidence reused: PASS")


# ------------------------------------------------------------
# STEP 2 — Identify exact opportunity types
#
# Two-plus includes both two-channel and three-channel days.
# Exactly-two and three-channel days are mutually exclusive.
# ------------------------------------------------------------

daily_093["opportunity_exactly_two"] = (
    daily_093["available_channel_count"].eq(2)
)

daily_093["opportunity_three"] = (
    daily_093["available_channel_count"].eq(3)
)

daily_093["opportunity_two_plus"] = (
    daily_093["available_channel_count"].ge(2)
)

assert (
    daily_093["opportunity_exactly_two"]
    | daily_093["opportunity_three"]
).eq(
    daily_093["opportunity_two_plus"]
).all()

assert not (
    daily_093["opportunity_exactly_two"]
    &
    daily_093["opportunity_three"]
).any()

# No opportunity is permitted on Forced Off-Days.
assert not daily_093.loc[
    daily_093["is_forced_offday"],
    "opportunity_two_plus",
].any()

# Only approved three-channel configurations can have
# three-channel opportunities.
assert daily_093.loc[
    daily_093["opportunity_three"],
    "channel_configuration",
].eq("E+U+H").all()

print("STEP 2 — Opportunity types identified: PASS")


# ------------------------------------------------------------
# STEP 3 — Identify naturally supported channel pairs
#
# Pair opportunities are NOT mutually exclusive:
# E+U+H availability supports all three channel pairs.
# ------------------------------------------------------------

PAIR_MASKS_093 = {
    "E+U": 3,
    "E+H": 5,
    "U+H": 6,
}

mask_values_093 = daily_093[
    "available_channel_mask"
].to_numpy(dtype=np.int64)

for pair, pair_mask in PAIR_MASKS_093.items():

    daily_093[f"pair_{pair.replace('+', '_')}"] = (
        (mask_values_093 & pair_mask) == pair_mask
    )

pair_cols_093 = {
    pair: f"pair_{pair.replace('+', '_')}"
    for pair in PAIR_MASKS_093
}

# Any two-plus-channel day must support at least one pair.
assert daily_093[
    list(pair_cols_093.values())
].any(axis=1).eq(
    daily_093["opportunity_two_plus"]
).all()

print("STEP 3 — Channel-pair opportunities mapped: PASS")


# ------------------------------------------------------------
# STEP 4 — Build per-attacker opportunity profile
# ------------------------------------------------------------

aggregation_093 = {
    "window_days": ("day", "size"),
    "active_days": ("is_active_day", "sum"),
    "forced_offdays": ("is_forced_offday", "sum"),
    "exactly_two_days": ("opportunity_exactly_two", "sum"),
    "three_channel_days": ("opportunity_three", "sum"),
    "two_plus_days": ("opportunity_two_plus", "sum"),
}

for pair, column in pair_cols_093.items():
    aggregation_093[
        f"pair_{pair.replace('+', '_')}_days"
    ] = (column, "sum")

user_opportunities_093 = (
    daily_093
    .groupby("user")
    .agg(**aggregation_093)
    .reset_index()
)

assignment_metadata_093 = assignments_093[
    [
        "user",
        "scenario_option_id",
        "window_workdays",
        "channel_configuration",
        "switching_level",
        "stealth_level",
    ]
].copy()

synchronization_user_profile_09 = (
    assignment_metadata_093.merge(
        user_opportunities_093,
        on="user",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

assert synchronization_user_profile_09[
    "_merge"
].eq("both").all()

synchronization_user_profile_09 = (
    synchronization_user_profile_09
    .drop(columns="_merge")
)

assert len(synchronization_user_profile_09) == 400
assert synchronization_user_profile_09["user"].is_unique

synchronization_user_profile_09[
    "has_two_plus_opportunity"
] = synchronization_user_profile_09[
    "two_plus_days"
].gt(0)

synchronization_user_profile_09[
    "has_three_channel_opportunity"
] = synchronization_user_profile_09[
    "three_channel_days"
].gt(0)

synchronization_user_profile_09[
    "two_plus_share_of_active_days"
] = (
    synchronization_user_profile_09["two_plus_days"]
    /
    synchronization_user_profile_09["active_days"]
)

synchronization_user_profile_09[
    "three_share_of_active_days"
] = (
    synchronization_user_profile_09["three_channel_days"]
    /
    synchronization_user_profile_09["active_days"]
)

assert np.isfinite(
    synchronization_user_profile_09[
        [
            "two_plus_share_of_active_days",
            "three_share_of_active_days",
        ]
    ].to_numpy(dtype=float)
).all()

print("STEP 4 — Per-attacker profiles constructed: PASS")


# ------------------------------------------------------------
# STEP 5 — Analyze user-level synchronization coverage
# ------------------------------------------------------------

two_plus_users_093 = int(
    synchronization_user_profile_09[
        "has_two_plus_opportunity"
    ].sum()
)

without_two_plus_093 = (
    400 - two_plus_users_093
)

triple_users_093 = (
    synchronization_user_profile_09.loc[
        synchronization_user_profile_09[
            "channel_configuration"
        ].eq("E+U+H")
    ]
)

assert len(triple_users_093) == 139

triple_capable_users_093 = int(
    triple_users_093[
        "has_three_channel_opportunity"
    ].sum()
)

triple_without_opportunity_093 = (
    139 - triple_capable_users_093
)

user_coverage_summary_09 = pd.DataFrame([
    {
        "metric": "Total approved attackers",
        "users": 400,
    },
    {
        "metric": "Users with 2+ channel opportunities",
        "users": two_plus_users_093,
    },
    {
        "metric": "Users without 2+ channel opportunities",
        "users": without_two_plus_093,
    },
    {
        "metric": "E+U+H assigned attackers",
        "users": 139,
    },
    {
        "metric": "E+U+H users with 3-channel opportunities",
        "users": triple_capable_users_093,
    },
    {
        "metric": "E+U+H users without 3-channel opportunities",
        "users": triple_without_opportunity_093,
    },
])

print("\nUSER-LEVEL SYNCHRONIZATION COVERAGE")
display(user_coverage_summary_09)

print("STEP 5 — User-level opportunity coverage: PASS")


# ------------------------------------------------------------
# STEP 6 — Summarize by channel configuration
# ------------------------------------------------------------

synchronization_by_channel_09 = (
    synchronization_user_profile_09
    .groupby("channel_configuration")
    .agg(
        attackers=("user", "size"),
        users_with_two_plus=(
            "has_two_plus_opportunity", "sum"
        ),
        users_with_three=(
            "has_three_channel_opportunity", "sum"
        ),
        total_active_days=("active_days", "sum"),
        exactly_two_days=("exactly_two_days", "sum"),
        three_channel_days=("three_channel_days", "sum"),
        two_plus_days=("two_plus_days", "sum"),
        median_two_plus_per_user=("two_plus_days", "median"),
        maximum_two_plus_per_user=("two_plus_days", "max"),
    )
    .reindex(CHANNEL_TARGETS_09)
    .reset_index()
)

synchronization_by_channel_09[
    "share_users_with_two_plus_pct"
] = (
    100
    * synchronization_by_channel_09["users_with_two_plus"]
    / synchronization_by_channel_09["attackers"]
).round(2)

print("\nSYNCHRONIZATION OPPORTUNITIES BY CHANNEL")
display(synchronization_by_channel_09)

print("STEP 6 — Channel-level analysis: PASS")


# ------------------------------------------------------------
# STEP 7 — Compare duration and Switching level
# ------------------------------------------------------------

synchronization_by_duration_09 = (
    synchronization_user_profile_09
    .groupby("window_workdays")
    .agg(
        attackers=("user", "size"),
        users_with_two_plus=(
            "has_two_plus_opportunity", "sum"
        ),
        users_with_three=(
            "has_three_channel_opportunity", "sum"
        ),
        active_days=("active_days", "sum"),
        two_plus_days=("two_plus_days", "sum"),
        three_channel_days=("three_channel_days", "sum"),
        median_two_plus_per_user=("two_plus_days", "median"),
    )
    .reindex(DURATION_TARGETS_09)
    .reset_index()
)

synchronization_by_duration_09[
    "two_plus_pct_of_active_days"
] = (
    100
    * synchronization_by_duration_09["two_plus_days"]
    / synchronization_by_duration_09["active_days"]
).round(2)

synchronization_by_switching_09 = (
    synchronization_user_profile_09
    .groupby("switching_level")
    .agg(
        attackers=("user", "size"),
        users_with_two_plus=(
            "has_two_plus_opportunity", "sum"
        ),
        users_with_three=(
            "has_three_channel_opportunity", "sum"
        ),
        active_days=("active_days", "sum"),
        two_plus_days=("two_plus_days", "sum"),
        three_channel_days=("three_channel_days", "sum"),
    )
    .reindex(SWITCHING_TARGETS_09)
    .reset_index()
)

synchronization_by_switching_09[
    "two_plus_pct_of_active_days"
] = (
    100
    * synchronization_by_switching_09["two_plus_days"]
    / synchronization_by_switching_09["active_days"]
).round(2)

print("\nOPPORTUNITIES BY ATTACK DURATION")
display(synchronization_by_duration_09)

print("\nOPPORTUNITIES BY SWITCHING LEVEL")
display(synchronization_by_switching_09)

print("STEP 7 — Cross-dimensional analysis: PASS")


# ------------------------------------------------------------
# STEP 8 — Analyze concentration among attackers
#
# Descriptive only. No quotas or rankings for assignment.
# ------------------------------------------------------------

ranked_users_093 = (
    synchronization_user_profile_09
    .sort_values(
        ["two_plus_days", "user"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

total_two_plus_093 = int(
    ranked_users_093["two_plus_days"].sum()
)

top_10_users_093 = ranked_users_093.head(10)

top_10_percent_users_093 = ranked_users_093.head(40)

concentration_summary_09 = pd.DataFrame([
    {
        "metric": "Total two-plus opportunity days",
        "value": total_two_plus_093,
    },
    {
        "metric": "Maximum opportunities for one user",
        "value": int(
            ranked_users_093["two_plus_days"].max()
        ),
    },
    {
        "metric": "Median opportunities per user",
        "value": float(
            ranked_users_093["two_plus_days"].median()
        ),
    },
    {
        "metric": "Top 10 users' share (%)",
        "value": round(
            100 * top_10_users_093["two_plus_days"].sum()
            / total_two_plus_093,
            2,
        ),
    },
    {
        "metric": "Top 10% of users' share (%)",
        "value": round(
            100
            * top_10_percent_users_093["two_plus_days"].sum()
            / total_two_plus_093,
            2,
        ),
    },
])

print("\nOPPORTUNITY CONCENTRATION")
display(concentration_summary_09)

print("STEP 8 — Opportunity concentration analyzed: PASS")


# ------------------------------------------------------------
# STEP 9 — Summarize supported channel pairs
#
# Pair counts overlap on E+U+H days.
# Do not sum these to obtain unique opportunity days.
# ------------------------------------------------------------

pair_opportunity_summary_09 = pd.DataFrame([
    {
        "channel_pair": pair,
        "opportunity_days": int(daily_093[column].sum()),
        "users_with_opportunity": int(
            daily_093.loc[
                daily_093[column], "user"
            ].nunique()
        ),
    }
    for pair, column in pair_cols_093.items()
])

print("\nNATURALLY SUPPORTED CHANNEL PAIRS")
display(pair_opportunity_summary_09)

print("STEP 9 — Channel-pair opportunities: PASS")


# ------------------------------------------------------------
# STEP 10 — Validate against Section 9.2 evidence
# ------------------------------------------------------------

expected_two_plus_093 = int(
    daily_window_09[
        "potential_two_channel_day"
    ].sum()
)

expected_three_093 = int(
    daily_window_09[
        "potential_three_channel_day"
    ].sum()
)

observed_two_plus_093 = int(
    daily_093["opportunity_two_plus"].sum()
)

observed_three_093 = int(
    daily_093["opportunity_three"].sum()
)

observed_exact_two_093 = int(
    daily_093["opportunity_exactly_two"].sum()
)

qa_checks_093 = {
    "Exactly 400 unique attacker profiles":
        len(synchronization_user_profile_09) == 400
        and synchronization_user_profile_09["user"].is_unique,

    "Two-plus days match Section 9.2":
        observed_two_plus_093 == expected_two_plus_093,

    "Three-channel days match Section 9.2":
        observed_three_093 == expected_three_093,

    "Exactly-two plus three reconciles":
        observed_exact_two_093 + observed_three_093
        == observed_two_plus_093,

    "No opportunities on Forced Off-Days":
        not daily_093.loc[
            daily_093["is_forced_offday"],
            "opportunity_two_plus",
        ].any(),

    "Three-channel opportunities only in E+U+H":
        daily_093.loc[
            daily_093["opportunity_three"],
            "channel_configuration",
        ].eq("E+U+H").all(),

    "Per-user counts reconcile":
        int(
            synchronization_user_profile_09[
                "two_plus_days"
            ].sum()
        ) == observed_two_plus_093
        and int(
            synchronization_user_profile_09[
                "three_channel_days"
            ].sum()
        ) == observed_three_093,

    "Channel summaries reconcile":
        int(
            synchronization_by_channel_09[
                "two_plus_days"
            ].sum()
        ) == observed_two_plus_093,

    "Duration summaries reconcile":
        int(
            synchronization_by_duration_09[
                "two_plus_days"
            ].sum()
        ) == observed_two_plus_093,

    "Switching summaries reconcile":
        int(
            synchronization_by_switching_09[
                "two_plus_days"
            ].sum()
        ) == observed_two_plus_093,
}

synchronization_opportunity_qa_09 = pd.DataFrame([
    {
        "check": check,
        "status": "PASS" if bool(passed) else "FAIL",
    }
    for check, passed in qa_checks_093.items()
])

print("\nSECTION 9.3 QUALITY ASSURANCE")
display(synchronization_opportunity_qa_09)

if not synchronization_opportunity_qa_09[
    "status"
].eq("PASS").all():
    raise RuntimeError(
        "Section 9.3 synchronization opportunity QA failed."
    )

print("STEP 10 — Opportunity analysis QA: PASS")


# ------------------------------------------------------------
# STEP 11 — Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NOTEBOOK 09 — SECTION 9.3 SUMMARY")
print("=" * 65)

print("Approved attackers              :", 400)
print("Users with 2+ channel opportunities:", two_plus_users_093)
print("Users without 2+ opportunities  :", without_two_plus_093)
print("E+U+H attackers                :", 139)
print("Users with 3-channel opportunities:", triple_capable_users_093)
print("E+U+H without triple opportunity:", triple_without_opportunity_093)
print("Exactly two-channel opportunity days:", observed_exact_two_093)
print("Three-channel opportunity days :", observed_three_093)
print("Two-plus opportunity days      :", observed_two_plus_093)
print(
    "Opportunity QA                :",
    f"{len(synchronization_opportunity_qa_09)}/"
    f"{len(synchronization_opportunity_qa_09)} PASS"
)
print("Synchronization days assigned  : NO")
print("Channel sequences constructed  : NO")
print("Approved scenarios modified    : NO")
print("Original files modified        : NO")
print("Files exported                 : NO")
print("=" * 65)

print("SECTION 9.3 — PASS")


STEP 1 — Approved daily evidence reused: PASS
STEP 2 — Opportunity types identified: PASS
STEP 3 — Channel-pair opportunities mapped: PASS
STEP 4 — Per-attacker profiles constructed: PASS

USER-LEVEL SYNCHRONIZATION COVERAGE


,metric,users
0,Total approved attackers,400
1,Users with 2+ channel opportunities,383
2,Users without 2+ channel opportunities,17
3,E+U+H assigned attackers,139
4,E+U+H users with 3-channel opportunities,83
5,E+U+H users without 3-channel opportunities,56


STEP 5 — User-level opportunity coverage: PASS

SYNCHRONIZATION OPPORTUNITIES BY CHANNEL


,channel_configuration,attackers,users_with_two_plus,users_with_three,total_active_days,exactly_two_days,three_channel_days,two_plus_days,median_two_plus_per_user,maximum_two_plus_per_user,share_users_with_two_plus_pct
0,E+U,117,112,0,2949,963,0,963,8.0,23,95.73
1,E+H,117,109,0,2703,633,0,633,5.0,15,93.16
2,U+H,27,26,0,223,50,0,50,1.0,9,96.30
3,E+U+H,139,136,83,3205,989,173,1162,8.0,27,97.84


STEP 6 — Channel-level analysis: PASS

OPPORTUNITIES BY ATTACK DURATION


,window_workdays,attackers,users_with_two_plus,users_with_three,active_days,two_plus_days,three_channel_days,median_two_plus_per_user,two_plus_pct_of_active_days
0,5,50,39,1,250,65,1,1.0,26.00
1,10,70,64,10,610,164,11,2.0,26.89
2,20,70,70,23,1215,436,37,6.0,35.88
3,30,70,70,16,1775,585,36,7.5,32.96
4,40,70,70,19,2335,681,44,9.0,29.16
5,50,70,70,14,2895,877,44,11.5,30.29


... (5 outputs left out)

,channel_pair,opportunity_days,users_with_opportunity
0,E+U,1510,239
1,E+H,1348,236
2,U+H,296,128


STEP 9 — Channel-pair opportunities: PASS

SECTION 9.3 QUALITY ASSURANCE


,check,status
0,Exactly 400 unique attacker profiles,PASS
1,Two-plus days match Section 9.2,PASS
2,Three-channel days match Section 9.2,PASS
3,Exactly-two plus three reconciles,PASS
4,No opportunities on Forced Off-Days,PASS
5,Three-channel opportunities only in E+U+H,PASS
6,Per-user counts reconcile,PASS
7,Channel summaries reconcile,PASS
8,Duration summaries reconcile,PASS
9,Switching summaries reconcile,PASS


STEP 10 — Opportunity analysis QA: PASS

NOTEBOOK 09 — SECTION 9.3 SUMMARY
Approved attackers              : 400
Users with 2+ channel opportunities: 383
Users without 2+ opportunities  : 17
E+U+H attackers                : 139
Users with 3-channel opportunities: 83
E+U+H without triple opportunity: 56
Exactly two-channel opportunity days: 2635
Three-channel opportunity days : 173
Two-plus opportunity days      : 2808
Opportunity QA                : 10/10 PASS
Synchronization days assigned  : NO
Channel sequences constructed  : NO
Approved scenarios modified    : NO
Original files modified        : NO
Files exported                 : NO
SECTION 9.3 — PASS



## 9.4 — Joint Switching & Synchronization Feasibility

### Objective

Determine which same-day channel synchronization configurations can be scheduled jointly with the approved Primary-Channel Switching requirements for each of the 400 attacker scenarios.

This section evaluates daily scheduling feasibility without selecting synchronization targets or constructing final channel sequences.

### Inputs

**Section 9.2 — Daily Window Reconstruction**

- `daily_window_09`: Approved daily timeline, Forced Off-Day positions, and naturally available channels.

**Section 9.3 — Synchronization Opportunity Analysis**

- `synchronization_user_profile_09`: Per-attacker opportunities for two-channel and three-channel synchronization.

**Notebook 08 — Approved Assignments**

- `approved_assignments_09`: Immutable attacker assignments and approved switching feasibility bitmasks.

### Scheduling Definitions

**Primary Channel**

Exactly one naturally available, approved channel is designated as the Primary Channel on each attack-active day.

**Primary-Channel Switch**

A switch occurs when the Primary Channel differs from the Primary Channel on the preceding attack-active day.

Forced Off-Days are skipped when counting transitions and do not create switches.

**Exactly Two-Channel Synchronization**

An attack-active day with exactly two distinct injected channels, including its Primary Channel.

**Three-Channel Synchronization**

An attack-active day with Email, USB, and HTTP used together, including one designated Primary Channel.

**Single-Channel Day**

An attack-active day with only its Primary Channel.

A synchronization day does not automatically count as a Primary-Channel Switch.

### Approved Constraints

1. Preserve all 400 approved attacker assignments.
2. Preserve every approved attack window and Forced Off-Day.
3. Every approved active day must receive a Primary Channel.
4. Primary and synchronized channels must belong to the approved channel configuration and be naturally available on the corresponding day.
5. All channels in the approved configuration must appear as Primary Channels during the attack window.
6. Exact Primary-Channel switch counts must be supported by the approved `set_switch_bitmask` from Notebook 08.
7. Exactly-two-channel and three-channel synchronization are mutually exclusive on a single day.
8. Zero synchronization must remain a valid option when supported by the approved switching evidence.
9. Synchronization opportunities cannot introduce additional active days or modify the approved assignments.

### Methodology

Use exact dynamic programming (DP) to evaluate joint scheduling feasibility.

The DP tracks:

- Channels already represented as Primary Channels.
- The previous Primary Channel.
- Attainable Primary-Channel switch counts.
- Number of exactly-two-channel synchronization days.
- Number of three-channel synchronization days.

For each attack-active day, the DP explores only naturally available and approved channel combinations.

Forced Off-Days preserve the previous Primary Channel without generating additional switches.

An option is jointly feasible only if it provides complete Primary-Channel coverage and at least one attainable switch count included in the previously approved switching bitmask.

### Feasibility Output

Each user receives a set of feasible count combinations:

`(exactly_two_channel_days, three_channel_days)`

Each combination includes the corresponding attainable switch-count bitmask.

The analysis also reports:

- Users with at least one feasible synchronization option.
- Users without feasible nonzero synchronization.
- Users with feasible three-channel synchronization.
- Number of feasible and infeasible count combinations.
- Feasibility by duration and channel configuration.

### Validation

- All 400 attackers retain at least one feasible daily scheduling option.
- The zero-synchronization case remains consistent with approved switching evidence.
- Every feasible option has a nonempty approved switching-count intersection.
- No synchronization option exceeds observed natural opportunities.
- No scheduling decisions are finalized.

### Methodological Boundaries

The approved switching requirements are reused through their existing bitmasks. This section does not redesign switching levels, recalculate the Behavioral Baseline, or regenerate the scenario catalog.

Only the additional joint feasibility of switching and synchronization is evaluated.

### Expected Outcome

An exact, per-attacker feasibility catalog of synchronization count combinations compatible with approved switching constraints.

Final synchronization target selection and daily sequence construction will follow in subsequent sections.

No source files are modified or exported.


In [4]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.4
# JOINT SWITCHING & SYNCHRONIZATION FEASIBILITY
#
# Exact DP | No target selection | No daily scheduling
# No injection | No exports
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Verify existing approved evidence
# ------------------------------------------------------------

if not daily_window_qa_09["status"].eq("PASS").all():
    raise RuntimeError("Section 9.2 QA must pass.")

if not synchronization_opportunity_qa_09[
    "status"
].eq("PASS").all():
    raise RuntimeError("Section 9.3 QA must pass.")

approved_094 = (
    approved_assignments_09
    .copy(deep=True)
)

timeline_094 = (
    daily_window_09
    .sort_values(["user", "day"])
    .copy()
)

profile_094 = (
    synchronization_user_profile_09
    .copy(deep=True)
)

assert len(approved_094) == 400
assert approved_094["user"].is_unique

assert len(profile_094) == 400
assert profile_094["user"].is_unique

assert not timeline_094.duplicated(
    ["user", "day"]
).any()

assert timeline_094["available_channel_mask"].notna().all()
assert approved_094["set_switch_bitmask"].notna().all()

CHANNEL_BITS_094 = (1, 2, 4)

CONFIG_MASKS_094 = {
    "E+U": 3,
    "E+H": 5,
    "U+H": 6,
    "E+U+H": 7,
}

print("STEP 1 — Approved evidence verified: PASS")


# ------------------------------------------------------------
# STEP 2 — Prepare per-attacker daily availability
#
# Reuse the already reconstructed daily timeline.
# No workday or Forced Off-Day recalculation.
# ------------------------------------------------------------

daily_masks_by_user_094 = {
    str(user): group["available_channel_mask"]
    .to_numpy(dtype=np.uint8)
    for user, group in timeline_094.groupby(
        "user", sort=False
    )
}

approved_094["user"] = approved_094[
    "user"
].astype("string")

profile_094["user"] = profile_094[
    "user"
].astype("string")

for row in approved_094.itertuples(index=False):

    user = str(row.user)

    if user not in daily_masks_by_user_094:
        raise RuntimeError(
            f"Missing daily window for user {user}"
        )

    masks = daily_masks_by_user_094[user]

    if len(masks) != int(row.window_workdays):
        raise RuntimeError(
            f"Window length mismatch: {user}"
        )

    actual_active = int(np.count_nonzero(masks))

    if actual_active != int(row.proposed_active_days):
        raise RuntimeError(
            f"Approved active-day mismatch: {user}"
        )

    if any(
        (int(mask) & ~CONFIG_MASKS_094[
            str(row.channel_configuration)
        ]) != 0
        for mask in masks
    ):
        raise RuntimeError(
            f"Invalid channel availability: {user}"
        )

print("STEP 2 — Per-attacker daily masks prepared: PASS")


# ------------------------------------------------------------
# STEP 3 — Exact joint dynamic programming
#
# DP state key:
#   (used_primary_mask, previous_primary)
#
# DP state value:
#   Packed bitset of achievable:
#       (switch_count, exactly_two_days, three_days)
#
# No synchronization targets are selected here.
# ------------------------------------------------------------

def joint_feasibility_dp_094(
    daily_masks,
    configuration_mask,
    max_two,
    max_three,
    max_switch,
):

    daily_masks = [
        int(mask)
        for mask in daily_masks
    ]

    configuration_mask = int(configuration_mask)

    max_two = int(max_two)
    max_three = int(max_three)
    max_switch = int(max_switch)

    # Width of the switch-count dimension.
    width = max_switch + 1

    two_stride = width

    three_stride = width * (max_two + 1)

    switch_block_mask = (1 << width) - 1

    # Precompute bitsets that prevent overflow
    # between adjacent count categories.
    safe_switch = 0
    safe_two = 0
    safe_three = 0

    for three in range(max_three + 1):

        for two in range(max_two + 1):

            offset = (
                three * three_stride
                + two * two_stride
            )

            # A switch increment is permitted only
            # when it stays within the switch block.
            if width > 1:
                safe_switch |= (
                    ((1 << (width - 1)) - 1)
                    << offset
                )

            # Increment exactly-two-day count.
            if two < max_two:
                safe_two |= (
                    switch_block_mask << offset
                )

            # Increment three-channel-day count.
            if three < max_three:
                safe_three |= (
                    switch_block_mask << offset
                )

    # Initial state:
    # no primary channel, no previous primary,
    # zero switches and zero synchronization days.
    states = {
        (0, 0): 1
    }

    action_cache = {}

    for available in daily_masks:

        # Forced Off-Day:
        # Preserve state without adding a switch
        # or any synchronized channel.
        if available == 0:
            continue

        available &= configuration_mask

        if available == 0:
            raise RuntimeError(
                "Active day has no approved channel."
            )

        if available not in action_cache:

            actions = set()

            # Enumerate approved nonempty channel subsets.
            for injection_mask in range(1, 8):

                if (
                    injection_mask & available
                ) != injection_mask:
                    continue

                channel_count = (
                    injection_mask.bit_count()
                )

                if channel_count == 2 and max_two == 0:
                    continue

                if channel_count == 3 and max_three == 0:
                    continue

                # The primary is one of the
                # channels in the injected subset.
                for primary in CHANNEL_BITS_094:

                    if injection_mask & primary:

                        actions.add((
                            primary,
                            channel_count,
                        ))

            action_cache[available] = sorted(actions)

        next_states = {}

        for (used, previous), bits in states.items():

            for primary, channel_count in (
                action_cache[available]
            ):

                new_used = used | primary

                # No switch on the first active day.
                # No switch if primary remains unchanged.
                if previous == 0 or previous == primary:

                    new_bits = bits

                else:

                    new_bits = (
                        bits & safe_switch
                    ) << 1

                if not new_bits:
                    continue

                if channel_count == 2:

                    new_bits = (
                        new_bits & safe_two
                    ) << two_stride

                elif channel_count == 3:

                    new_bits = (
                        new_bits & safe_three
                    ) << three_stride

                if not new_bits:
                    continue

                key = (new_used, primary)

                next_states[key] = (
                    next_states.get(key, 0)
                    | new_bits
                )

        states = next_states

        if not states:
            break

    # Full Primary-Channel coverage is mandatory.
    final_bits = 0

    for (used, previous), bits in states.items():

        if used == configuration_mask:
            final_bits |= bits

    # Extract a switch-count mask for every
    # exact synchronization-count combination.
    results = {}

    for three in range(max_three + 1):

        for two in range(max_two + 1):

            offset = (
                three * three_stride
                + two * two_stride
            )

            switch_mask = (
                final_bits >> offset
            ) & switch_block_mask

            results[(two, three)] = int(
                switch_mask
            )

    return results


print("STEP 3 — Exact joint DP prepared: PASS")


# ------------------------------------------------------------
# STEP 4 — Evaluate approved scenarios
#
# Use natural opportunities as bounds.
# Use approved switching bitmasks as constraints.
#
# Do NOT introduce arbitrary synchronization quotas.
# ------------------------------------------------------------

opportunity_lookup_094 = (
    profile_094
    .set_index("user")
)

joint_rows_094 = []
zero_proof_rows_094 = []

for row in approved_094.itertuples(index=False):

    user = str(row.user)

    daily_masks = daily_masks_by_user_094[user]

    active_days = int(row.proposed_active_days)

    configuration_mask = CONFIG_MASKS_094[
        str(row.channel_configuration)
    ]

    approved_switch_mask = int(
        row.set_switch_bitmask
    )

    if approved_switch_mask <= 0:
        raise RuntimeError(
            f"Missing approved switching proof: {user}"
        )

    # An exact switch count cannot exceed
    # the number of active-day transitions.
    max_possible_switch = active_days - 1

    allowed_range_mask = (
        (1 << (max_possible_switch + 1)) - 1
    )

    if approved_switch_mask & ~allowed_range_mask:
        raise RuntimeError(
            f"Switch mask exceeds active transitions: {user}"
        )

    maximum_switch = (
        approved_switch_mask.bit_length() - 1
    )

    # Natural upper bounds.
    max_two = int(
        opportunity_lookup_094.loc[
            user, "two_plus_days"
        ]
    )

    max_three = int(
        opportunity_lookup_094.loc[
            user, "three_channel_days"
        ]
    )

    assert max_two <= active_days
    assert max_three <= max_two

    exact_results = joint_feasibility_dp_094(
        daily_masks=daily_masks,
        configuration_mask=configuration_mask,
        max_two=max_two,
        max_three=max_three,
        max_switch=maximum_switch,
    )

    # Zero synchronization must retain at least
    # the previously approved switch-count evidence.
    zero_switch_mask = (
        exact_results[(0, 0)]
        & approved_switch_mask
    )

    zero_proof_rows_094.append({
        "user": user,
        "approved_switch_mask": approved_switch_mask,
        "zero_sync_switch_mask": zero_switch_mask,
        "proof_preserved": (
            zero_switch_mask == approved_switch_mask
        ),
    })

    for (two, three), possible_mask in (
        exact_results.items()
    ):

        supported_mask = (
            possible_mask
            & approved_switch_mask
        )

        joint_rows_094.append({
            "user": user,
            "scenario_option_id": int(
                row.scenario_option_id
            ),
            "window_workdays": int(
                row.window_workdays
            ),
            "channel_configuration": str(
                row.channel_configuration
            ),
            "switching_level": str(
                row.switching_level
            ),
            "target_2ch_days": int(two),
            "target_3ch_days": int(three),
            "total_sync_days": int(two + three),
            "joint_feasible": bool(
                supported_mask
            ),
            "attainable_switch_mask": int(
                supported_mask
            ),
        })

same_day_joint_feasibility_09 = pd.DataFrame(
    joint_rows_094
)

zero_proof_audit_09 = pd.DataFrame(
    zero_proof_rows_094
)

if not zero_proof_audit_09[
    "proof_preserved"
].all():

    print("\nSWITCHING PROOF MISMATCHES")
    display(
        zero_proof_audit_09.loc[
            ~zero_proof_audit_09["proof_preserved"]
        ]
    )

    raise RuntimeError(
        "Joint DP contradicts approved zero-"
        "synchronization switching evidence."
    )

print("STEP 4 — Exact joint feasibility evaluated: PASS")
print(
    "Evaluated combinations:",
    f"{len(same_day_joint_feasibility_09):,}"
)


# ------------------------------------------------------------
# STEP 5 — Build user-level feasibility profile
# ------------------------------------------------------------

feasible_094 = same_day_joint_feasibility_09.loc[
    same_day_joint_feasibility_09[
        "joint_feasible"
    ]
].copy()

nonzero_feasible_094 = feasible_094.loc[
    feasible_094["total_sync_days"].gt(0)
].copy()

triple_feasible_094 = feasible_094.loc[
    feasible_094["target_3ch_days"].gt(0)
].copy()

summary_by_user_094 = (
    same_day_joint_feasibility_09
    .groupby("user")
    .agg(
        tested_combinations=("joint_feasible", "size"),
        feasible_combinations=("joint_feasible", "sum"),
    )
    .reset_index()
)

nonzero_user_counts_094 = (
    nonzero_feasible_094
    .groupby("user")
    .size()
    .rename("feasible_nonzero_combinations")
)

triple_user_counts_094 = (
    triple_feasible_094
    .groupby("user")
    .size()
    .rename("feasible_triple_combinations")
)

joint_feasibility_user_09 = (
    summary_by_user_094
    .merge(
        nonzero_user_counts_094,
        on="user",
        how="left",
    )
    .merge(
        triple_user_counts_094,
        on="user",
        how="left",
    )
)

for col in [
    "feasible_nonzero_combinations",
    "feasible_triple_combinations",
]:

    joint_feasibility_user_09[col] = (
        joint_feasibility_user_09[col]
        .fillna(0)
        .astype(int)
    )

joint_feasibility_user_09[
    "has_jointly_feasible_sync"
] = (
    joint_feasibility_user_09[
        "feasible_nonzero_combinations"
    ].gt(0)
)

joint_feasibility_user_09[
    "has_jointly_feasible_triple"
] = (
    joint_feasibility_user_09[
        "feasible_triple_combinations"
    ].gt(0)
)

joint_feasibility_user_09 = (
    approved_094[
        [
            "user",
            "window_workdays",
            "channel_configuration",
            "switching_level",
        ]
    ]
    .merge(
        joint_feasibility_user_09,
        on="user",
        how="left",
        validate="one_to_one",
    )
)

assert len(joint_feasibility_user_09) == 400
assert joint_feasibility_user_09["user"].is_unique

print("STEP 5 — User-level feasibility profile: PASS")


# ------------------------------------------------------------
# STEP 6 — Summarize joint feasibility
# ------------------------------------------------------------

joint_feasibility_summary_09 = (
    joint_feasibility_user_09
    .groupby(
        ["window_workdays", "channel_configuration"],
        observed=True,
    )
    .agg(
        attackers=("user", "size"),
        users_with_feasible_sync=(
            "has_jointly_feasible_sync",
            "sum",
        ),
        users_with_feasible_triple=(
            "has_jointly_feasible_triple",
            "sum",
        ),
        total_feasible_combinations=(
            "feasible_combinations",
            "sum",
        ),
    )
    .reset_index()
)

feasibility_by_channel_09 = (
    joint_feasibility_user_09
    .groupby("channel_configuration")
    .agg(
        attackers=("user", "size"),
        users_with_feasible_sync=(
            "has_jointly_feasible_sync", "sum"
        ),
        users_with_feasible_triple=(
            "has_jointly_feasible_triple", "sum"
        ),
    )
    .reindex(CHANNEL_TARGETS_09)
    .reset_index()
)

feasibility_by_switching_09 = (
    joint_feasibility_user_09
    .groupby("switching_level")
    .agg(
        attackers=("user", "size"),
        users_with_feasible_sync=(
            "has_jointly_feasible_sync", "sum"
        ),
        users_with_feasible_triple=(
            "has_jointly_feasible_triple", "sum"
        ),
    )
    .reindex(SWITCHING_TARGETS_09)
    .reset_index()
)

print("\nJOINT FEASIBILITY BY CHANNEL")
display(feasibility_by_channel_09)

print("\nJOINT FEASIBILITY BY SWITCHING")
display(feasibility_by_switching_09)

print("\nJOINT FEASIBILITY BY DURATION × CHANNEL")
display(joint_feasibility_summary_09)

print("STEP 6 — Joint feasibility summarized: PASS")


# ------------------------------------------------------------
# STEP 7 — Quality assurance
# ------------------------------------------------------------

zero_options_094 = (
    same_day_joint_feasibility_09.loc[
        same_day_joint_feasibility_09[
            "total_sync_days"
        ].eq(0)
    ]
)

qa_checks_094 = {
    "400 approved attackers evaluated":
        joint_feasibility_user_09["user"].nunique() == 400,

    "Unique count-combination keys":
        not same_day_joint_feasibility_09.duplicated(
            ["user", "target_2ch_days", "target_3ch_days"]
        ).any(),

    "Every attacker has a feasible option":
        joint_feasibility_user_09[
            "feasible_combinations"
        ].gt(0).all(),

    "Zero synchronization feasible for all":
        len(zero_options_094) == 400
        and zero_options_094["joint_feasible"].all(),

    "Approved zero-sync proof preserved":
        zero_proof_audit_09[
            "proof_preserved"
        ].all(),

    "Feasible combinations have switching evidence":
        feasible_094[
            "attainable_switch_mask"
        ].gt(0).all(),

    "Three-channel sync only in E+U+H":
        triple_feasible_094[
            "channel_configuration"
        ].eq("E+U+H").all(),

    "Feasible sync coverage within natural opportunities":
        set(nonzero_feasible_094["user"]).issubset(
            set(
                profile_094.loc[
                    profile_094["two_plus_days"].gt(0),
                    "user",
                ]
            )
        ),

    "Feasible triple coverage within natural opportunities":
        set(triple_feasible_094["user"]).issubset(
            set(
                profile_094.loc[
                    profile_094["three_channel_days"].gt(0),
                    "user",
                ]
            )
        ),

    "All scenario summaries reconcile":
        int(
            joint_feasibility_summary_09[
                "attackers"
            ].sum()
        ) == 400,
}

joint_feasibility_qa_09 = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if bool(ok) else "FAIL",
    }
    for name, ok in qa_checks_094.items()
])

print("\nSECTION 9.4 QUALITY ASSURANCE")
display(joint_feasibility_qa_09)

if not joint_feasibility_qa_09[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Section 9.4 joint feasibility QA failed."
    )

print("STEP 7 — Joint feasibility QA: PASS")


# ------------------------------------------------------------
# STEP 8 — Final summary
# ------------------------------------------------------------

users_with_joint_sync_094 = int(
    joint_feasibility_user_09[
        "has_jointly_feasible_sync"
    ].sum()
)

users_with_joint_triple_094 = int(
    joint_feasibility_user_09[
        "has_jointly_feasible_triple"
    ].sum()
)

tested_options_094 = len(
    same_day_joint_feasibility_09
)

feasible_options_094 = int(
    same_day_joint_feasibility_09[
        "joint_feasible"
    ].sum()
)

print("\n" + "=" * 65)
print("NOTEBOOK 09 — SECTION 9.4 SUMMARY")
print("=" * 65)

print("Approved attackers              : 400")
print("Tested count combinations       :", tested_options_094)
print("Jointly feasible combinations   :", feasible_options_094)
print(
    "Infeasible combinations        :",
    tested_options_094 - feasible_options_094
)
print(
    "Users with feasible synchronization:",
    users_with_joint_sync_094
)
print(
    "Users with feasible triple synchronization:",
    users_with_joint_triple_094
)
print(
    "Users without feasible synchronization:",
    400 - users_with_joint_sync_094
)
print(
    "Approved switching proofs preserved:",
    int(zero_proof_audit_09["proof_preserved"].sum()),
    "/400"
)
print(
    "Joint feasibility QA           :",
    f"{len(joint_feasibility_qa_09)}/"
    f"{len(joint_feasibility_qa_09)} PASS"
)

print("Synchronization targets selected: NO")
print("Daily channel sequences created : NO")
print("Approved assignments modified   : NO")
print("Original files modified         : NO")
print("Files exported                  : NO")
print("=" * 65)

print("SECTION 9.4 — PASS")


STEP 1 — Approved evidence verified: PASS
STEP 2 — Per-attacker daily masks prepared: PASS
STEP 3 — Exact joint DP prepared: PASS

SWITCHING PROOF MISMATCHES


,user,approved_switch_mask,zero_sync_switch_mask,proof_preserved
17,AIP1614,3,2,False
22,AME1490,3,2,False
55,BSM2844,3,2,False
57,BVC0009,3,2,False
61,BYF3784,8191,8190,False
79,CMH2342,8191,8190,False
85,CRM0139,31,30,False
96,DCC2999,4095,4094,False
147,GNC2237,511,510,False
150,GSA3815,15,14,False


RuntimeError: Joint DP contradicts approved zero-synchronization switching evidence.

In [6]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.4
# JOINT SWITCHING & SYNCHRONIZATION FEASIBILITY
#
# Exact DP | Approved switching-level filtering
# No target selection | No daily scheduling
# No injection | No exports
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Verify approved inputs
# ------------------------------------------------------------

if not daily_window_qa_09["status"].eq("PASS").all():
    raise RuntimeError("Section 9.2 QA must pass.")

if not synchronization_opportunity_qa_09[
    "status"
].eq("PASS").all():
    raise RuntimeError("Section 9.3 QA must pass.")

approved_094 = (
    approved_assignments_09
    .copy(deep=True)
)

timeline_094 = (
    daily_window_09
    .sort_values(["user", "day"])
    .copy(deep=True)
)

profile_094 = (
    synchronization_user_profile_09
    .copy(deep=True)
)

for frame in [approved_094, timeline_094, profile_094]:
    frame["user"] = frame["user"].astype("string")

assert len(approved_094) == 400
assert approved_094["user"].is_unique

assert len(profile_094) == 400
assert profile_094["user"].is_unique

assert not timeline_094.duplicated(
    ["user", "day"]
).any()

assert timeline_094[
    "available_channel_mask"
].notna().all()

assert approved_094[
    "set_switch_bitmask"
].notna().all()

CHANNEL_BITS_094 = (1, 2, 4)

CONFIG_MASKS_094 = {
    "E+U": 3,
    "E+H": 5,
    "U+H": 6,
    "E+U+H": 7,
}

print("STEP 1 — Approved evidence verified: PASS")


# ------------------------------------------------------------
# STEP 2 — Prepare daily availability
#
# Reuse Section 9.2 results.
# No workday or Forced Off-Day recalculation.
# ------------------------------------------------------------

daily_masks_by_user_094 = {
    str(user): group[
        "available_channel_mask"
    ].to_numpy(dtype=np.uint8)

    for user, group in timeline_094.groupby(
        "user", sort=False
    )
}

for row in approved_094.itertuples(index=False):

    user = str(row.user)

    if user not in daily_masks_by_user_094:
        raise RuntimeError(
            f"Missing daily timeline: {user}"
        )

    masks = daily_masks_by_user_094[user]

    if len(masks) != int(row.window_workdays):
        raise RuntimeError(
            f"Window length mismatch: {user}"
        )

    active_days = int(np.count_nonzero(masks))

    if active_days != int(row.proposed_active_days):
        raise RuntimeError(
            f"Active-day mismatch: {user}"
        )

    configuration_mask = CONFIG_MASKS_094[
        str(row.channel_configuration)
    ]

    for mask in masks:
        if int(mask) & ~configuration_mask:
            raise RuntimeError(
                f"Invalid daily channel mask: {user}"
            )

print("STEP 2 — Daily availability prepared: PASS")


# ------------------------------------------------------------
# STEP 3 — Reuse approved Switching-level definitions
#
# The level masks restrict approved switch-count
# evidence to the applicable Low/Medium/High range.
#
# Special five-day targets follow the previously
# approved switching policy.
# ------------------------------------------------------------

SWITCH_RANGES_094 = {
    "Low": (10, 25),
    "Medium": (30, 45),
    "High": (50, 65),
}

FIVE_DAY_SWITCHES_094 = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
}


def allowed_switch_mask_094(
    duration,
    active_days,
    level,
):
    duration = int(duration)
    active_days = int(active_days)
    level = str(level)

    if level not in SWITCH_RANGES_094:
        raise ValueError(
            f"Unknown Switching level: {level}"
        )

    if duration == 5:

        if active_days != 5:
            raise ValueError(
                "Five-day scenarios must have "
                "five approved active days."
            )

        return 1 << FIVE_DAY_SWITCHES_094[level]

    transitions = active_days - 1

    low_pct, high_pct = SWITCH_RANGES_094[level]

    minimum = (
        transitions * low_pct + 99
    ) // 100

    maximum = (
        transitions * high_pct
    ) // 100

    return sum(
        1 << count
        for count in range(
            max(1, minimum),
            maximum + 1,
        )
    )


print("STEP 3 — Switching-level masks prepared: PASS")


# ------------------------------------------------------------
# STEP 4 — Exact joint dynamic programming
#
# State key:
#   (used_primary_mask, previous_primary)
#
# State value:
#   Packed bitset representing:
#       (switch_count, exactly_two_days, three_days)
#
# Synchronization does not automatically count
# as a Primary-Channel Switch.
# ------------------------------------------------------------

def joint_feasibility_dp_094(
    daily_masks,
    configuration_mask,
    max_two,
    max_three,
    max_switch,
):

    daily_masks = [
        int(mask)
        for mask in daily_masks
    ]

    configuration_mask = int(
        configuration_mask
    )

    max_two = int(max_two)
    max_three = int(max_three)
    max_switch = int(max_switch)

    width = max_switch + 1

    two_stride = width

    three_stride = (
        width * (max_two + 1)
    )

    switch_block_mask = (
        (1 << width) - 1
    )

    # Safe increment masks prevent count overflow
    # between packed bitset dimensions.

    safe_switch = 0
    safe_two = 0
    safe_three = 0

    for three in range(max_three + 1):

        for two in range(max_two + 1):

            offset = (
                three * three_stride
                + two * two_stride
            )

            if width > 1:
                safe_switch |= (
                    ((1 << (width - 1)) - 1)
                    << offset
                )

            if two < max_two:
                safe_two |= (
                    switch_block_mask << offset
                )

            if three < max_three:
                safe_three |= (
                    switch_block_mask << offset
                )

    # Initial state:
    # no Primary Channel, no previous channel,
    # no switching, and no synchronization.

    states = {
        (0, 0): 1
    }

    action_cache = {}

    for available in daily_masks:

        # Forced Off-Day:
        # no injection and no switching.
        if available == 0:
            continue

        available &= configuration_mask

        if available == 0:
            raise RuntimeError(
                "An active day has no approved channel."
            )

        if available not in action_cache:

            actions = set()

            # Enumerate available channel subsets.
            for injection_mask in range(1, 8):

                if (
                    injection_mask & available
                ) != injection_mask:
                    continue

                channel_count = (
                    injection_mask.bit_count()
                )

                if (
                    channel_count == 2
                    and max_two == 0
                ):
                    continue

                if (
                    channel_count == 3
                    and max_three == 0
                ):
                    continue

                # Primary Channel must belong to
                # the selected injection subset.
                for primary in CHANNEL_BITS_094:

                    if injection_mask & primary:

                        actions.add((
                            primary,
                            channel_count,
                        ))

            action_cache[available] = sorted(
                actions
            )

        next_states = {}

        for (used, previous), bits in (
            states.items()
        ):

            for primary, channel_count in (
                action_cache[available]
            ):

                new_used = used | primary

                # First active day is not a switch.
                # Same primary channel is not a switch.
                if (
                    previous == 0
                    or previous == primary
                ):

                    new_bits = bits

                else:

                    new_bits = (
                        bits & safe_switch
                    ) << 1

                if not new_bits:
                    continue

                # Exactly-two-channel synchronization.
                if channel_count == 2:

                    new_bits = (
                        new_bits & safe_two
                    ) << two_stride

                # Three-channel synchronization.
                elif channel_count == 3:

                    new_bits = (
                        new_bits & safe_three
                    ) << three_stride

                if not new_bits:
                    continue

                key = (
                    new_used,
                    primary,
                )

                next_states[key] = (
                    next_states.get(key, 0)
                    | new_bits
                )

        states = next_states

        if not states:
            break

    # Require full Primary-Channel coverage.
    final_bits = 0

    for (used, previous), bits in (
        states.items()
    ):

        if used == configuration_mask:
            final_bits |= bits

    # Extract attainable switch-count masks
    # for each exact synchronization combination.

    results = {}

    for three in range(max_three + 1):

        for two in range(max_two + 1):

            offset = (
                three * three_stride
                + two * two_stride
            )

            switch_mask = (
                final_bits >> offset
            ) & switch_block_mask

            results[(two, three)] = int(
                switch_mask
            )

    return results


print("STEP 4 — Exact joint DP prepared: PASS")


# ------------------------------------------------------------
# STEP 5 — Evaluate all approved attacker scenarios
#
# CORRECTION:
# Compare switching evidence WITHIN the approved
# Switching level, not against the entire raw bitmask.
# ------------------------------------------------------------

opportunity_lookup_094 = (
    profile_094.set_index("user")
)

joint_rows_094 = []
zero_proof_rows_094 = []

for row in approved_094.itertuples(index=False):

    user = str(row.user)

    daily_masks = (
        daily_masks_by_user_094[user]
    )

    active_days = int(
        row.proposed_active_days
    )

    configuration_mask = (
        CONFIG_MASKS_094[
            str(row.channel_configuration)
        ]
    )

    approved_switch_mask = int(
        row.set_switch_bitmask
    )

    if approved_switch_mask <= 0:
        raise RuntimeError(
            f"Missing approved switching proof: {user}"
        )

    max_possible_switch = active_days - 1

    if max_possible_switch < 0:
        raise RuntimeError(
            f"Invalid active-day count: {user}"
        )

    allowed_range_mask = (
        (1 << (max_possible_switch + 1)) - 1
    )

    if approved_switch_mask & ~allowed_range_mask:
        raise RuntimeError(
            f"Switch proof exceeds active transitions: {user}"
        )

    # Approved Low/Medium/High switch counts.
    allowed_level_mask = allowed_switch_mask_094(
        duration=row.window_workdays,
        active_days=row.proposed_active_days,
        level=row.switching_level,
    )

    # Only the switch counts relevant to the
    # approved level are required for scheduling.
    approved_level_mask = (
        approved_switch_mask
        & allowed_level_mask
    )

    if approved_level_mask == 0:
        raise RuntimeError(
            f"No approved switch count in level: {user}"
        )

    # Natural synchronization-opportunity bounds.
    max_two = int(
        opportunity_lookup_094.loc[
            user,
            "two_plus_days",
        ]
    )

    max_three = int(
        opportunity_lookup_094.loc[
            user,
            "three_channel_days",
        ]
    )

    assert max_two <= active_days
    assert max_three <= max_two

    # Only switch counts in the approved level
    # need to be represented by the DP.
    maximum_switch = (
        approved_level_mask.bit_length() - 1
    )

    exact_results = joint_feasibility_dp_094(
        daily_masks=daily_masks,
        configuration_mask=configuration_mask,
        max_two=max_two,
        max_three=max_three,
        max_switch=maximum_switch,
    )

    # Correct zero-synchronization comparison.
    zero_level_mask = (
        exact_results[(0, 0)]
        & approved_level_mask
    )

    zero_proof_rows_094.append({
        "user": user,
        "approved_switch_mask":
            approved_switch_mask,
        "allowed_level_mask":
            allowed_level_mask,
        "approved_level_mask":
            approved_level_mask,
        "zero_sync_switch_mask":
            zero_level_mask,
        "proof_preserved": (
            zero_level_mask == approved_level_mask
        ),
    })

    # Evaluate each synchronization-count option.
    for (two, three), possible_mask in (
        exact_results.items()
    ):

        supported_mask = (
            possible_mask
            & approved_level_mask
        )

        joint_rows_094.append({
            "user": user,
            "scenario_option_id": int(
                row.scenario_option_id
            ),
            "window_workdays": int(
                row.window_workdays
            ),
            "channel_configuration": str(
                row.channel_configuration
            ),
            "switching_level": str(
                row.switching_level
            ),
            "target_2ch_days": int(two),
            "target_3ch_days": int(three),
            "total_sync_days": int(
                two + three
            ),
            "joint_feasible": bool(
                supported_mask
            ),
            "attainable_switch_mask": int(
                supported_mask
            ),
        })

same_day_joint_feasibility_09 = pd.DataFrame(
    joint_rows_094
)

zero_proof_audit_09 = pd.DataFrame(
    zero_proof_rows_094
)

if not zero_proof_audit_09[
    "proof_preserved"
].all():

    print("\nZERO-SYNC SWITCHING MISMATCHES")

    display(
        zero_proof_audit_09.loc[
            ~zero_proof_audit_09[
                "proof_preserved"
            ]
        ]
    )

    raise RuntimeError(
        "Level-filtered switching evidence mismatch."
    )

print("STEP 5 — Joint feasibility evaluated: PASS")
print(
    "Evaluated combinations:",
    f"{len(same_day_joint_feasibility_09):,}"
)


# ------------------------------------------------------------
# STEP 6 — Build user-level feasibility profile
# ------------------------------------------------------------

feasible_094 = (
    same_day_joint_feasibility_09.loc[
        same_day_joint_feasibility_09[
            "joint_feasible"
        ]
    ].copy()
)

nonzero_feasible_094 = (
    feasible_094.loc[
        feasible_094[
            "total_sync_days"
        ].gt(0)
    ].copy()
)

triple_feasible_094 = (
    feasible_094.loc[
        feasible_094[
            "target_3ch_days"
        ].gt(0)
    ].copy()
)

summary_by_user_094 = (
    same_day_joint_feasibility_09
    .groupby("user")
    .agg(
        tested_combinations=(
            "joint_feasible", "size"
        ),
        feasible_combinations=(
            "joint_feasible", "sum"
        ),
    )
    .reset_index()
)

nonzero_counts_094 = (
    nonzero_feasible_094
    .groupby("user")
    .size()
    .rename(
        "feasible_nonzero_combinations"
    )
)

triple_counts_094 = (
    triple_feasible_094
    .groupby("user")
    .size()
    .rename(
        "feasible_triple_combinations"
    )
)

joint_feasibility_user_09 = (
    summary_by_user_094
    .merge(
        nonzero_counts_094,
        on="user",
        how="left",
    )
    .merge(
        triple_counts_094,
        on="user",
        how="left",
    )
)

for col in [
    "feasible_nonzero_combinations",
    "feasible_triple_combinations",
]:

    joint_feasibility_user_09[col] = (
        joint_feasibility_user_09[col]
        .fillna(0)
        .astype(int)
    )

joint_feasibility_user_09[
    "has_jointly_feasible_sync"
] = (
    joint_feasibility_user_09[
        "feasible_nonzero_combinations"
    ].gt(0)
)

joint_feasibility_user_09[
    "has_jointly_feasible_triple"
] = (
    joint_feasibility_user_09[
        "feasible_triple_combinations"
    ].gt(0)
)

joint_feasibility_user_09 = (
    approved_094[
        [
            "user",
            "window_workdays",
            "channel_configuration",
            "switching_level",
        ]
    ]
    .merge(
        joint_feasibility_user_09,
        on="user",
        how="left",
        validate="one_to_one",
    )
)

assert len(joint_feasibility_user_09) == 400
assert joint_feasibility_user_09[
    "user"
].is_unique

print("STEP 6 — User-level feasibility profile: PASS")


# ------------------------------------------------------------
# STEP 7 — Summarize joint feasibility
# ------------------------------------------------------------

joint_feasibility_summary_09 = (
    joint_feasibility_user_09
    .groupby(
        [
            "window_workdays",
            "channel_configuration",
        ],
        observed=True,
    )
    .agg(
        attackers=("user", "size"),

        users_with_feasible_sync=(
            "has_jointly_feasible_sync",
            "sum",
        ),

        users_with_feasible_triple=(
            "has_jointly_feasible_triple",
            "sum",
        ),

        total_feasible_combinations=(
            "feasible_combinations",
            "sum",
        ),
    )
    .reset_index()
)

feasibility_by_channel_09 = (
    joint_feasibility_user_09
    .groupby("channel_configuration")
    .agg(
        attackers=("user", "size"),

        users_with_feasible_sync=(
            "has_jointly_feasible_sync",
            "sum",
        ),

        users_with_feasible_triple=(
            "has_jointly_feasible_triple",
            "sum",
        ),
    )
    .reindex(CHANNEL_TARGETS_09)
    .reset_index()
)

feasibility_by_switching_09 = (
    joint_feasibility_user_09
    .groupby("switching_level")
    .agg(
        attackers=("user", "size"),

        users_with_feasible_sync=(
            "has_jointly_feasible_sync",
            "sum",
        ),

        users_with_feasible_triple=(
            "has_jointly_feasible_triple",
            "sum",
        ),
    )
    .reindex(SWITCHING_TARGETS_09)
    .reset_index()
)

print("\nJOINT FEASIBILITY BY CHANNEL")
display(feasibility_by_channel_09)

print("\nJOINT FEASIBILITY BY SWITCHING")
display(feasibility_by_switching_09)

print("\nJOINT FEASIBILITY BY DURATION × CHANNEL")
display(joint_feasibility_summary_09)

print("STEP 7 — Joint feasibility summarized: PASS")


# ------------------------------------------------------------
# STEP 8 — Consolidated quality assurance
# ------------------------------------------------------------

zero_options_094 = (
    same_day_joint_feasibility_09.loc[
        same_day_joint_feasibility_09[
            "total_sync_days"
        ].eq(0)
    ]
)

qa_checks_094 = {

    "400 approved attackers evaluated":
        joint_feasibility_user_09[
            "user"
        ].nunique() == 400,

    "Unique synchronization-count keys":
        not same_day_joint_feasibility_09.duplicated(
            [
                "user",
                "target_2ch_days",
                "target_3ch_days",
            ]
        ).any(),

    "Every attacker has a feasible option":
        joint_feasibility_user_09[
            "feasible_combinations"
        ].gt(0).all(),

    "Zero synchronization feasible for all":
        len(zero_options_094) == 400
        and zero_options_094[
            "joint_feasible"
        ].all(),

    "Approved level-filtered proof preserved":
        zero_proof_audit_09[
            "proof_preserved"
        ].all(),

    "Feasible options retain switching evidence":
        feasible_094[
            "attainable_switch_mask"
        ].gt(0).all(),

    "Three-channel synchronization only in E+U+H":
        triple_feasible_094[
            "channel_configuration"
        ].eq("E+U+H").all(),

    "Feasible synchronization within natural opportunities":
        set(
            nonzero_feasible_094["user"]
        ).issubset(
            set(
                profile_094.loc[
                    profile_094[
                        "two_plus_days"
                    ].gt(0),
                    "user",
                ]
            )
        ),

    "Feasible triple synchronization within natural opportunities":
        set(
            triple_feasible_094["user"]
        ).issubset(
            set(
                profile_094.loc[
                    profile_094[
                        "three_channel_days"
                    ].gt(0),
                    "user",
                ]
            )
        ),

    "Scenario summaries reconcile":
        int(
            joint_feasibility_summary_09[
                "attackers"
            ].sum()
        ) == 400,
}

joint_feasibility_qa_09 = pd.DataFrame([
    {
        "check": name,
        "status": (
            "PASS" if bool(passed) else "FAIL"
        ),
    }
    for name, passed in qa_checks_094.items()
])

print("\nSECTION 9.4 QUALITY ASSURANCE")
display(joint_feasibility_qa_09)

if not joint_feasibility_qa_09[
    "status"
].eq("PASS").all():

    raise RuntimeError(
        "Section 9.4 joint feasibility QA failed."
    )

print("STEP 8 — Joint feasibility QA: PASS")


# ------------------------------------------------------------
# STEP 9 — Final summary
# ------------------------------------------------------------

users_with_joint_sync_094 = int(
    joint_feasibility_user_09[
        "has_jointly_feasible_sync"
    ].sum()
)

users_with_joint_triple_094 = int(
    joint_feasibility_user_09[
        "has_jointly_feasible_triple"
    ].sum()
)

tested_options_094 = len(
    same_day_joint_feasibility_09
)

feasible_options_094 = int(
    same_day_joint_feasibility_09[
        "joint_feasible"
    ].sum()
)

print("\n" + "=" * 65)
print("NOTEBOOK 09 — SECTION 9.4 SUMMARY")
print("=" * 65)

print("Approved attackers                 : 400")

print(
    "Tested count combinations          :",
    tested_options_094
)

print(
    "Jointly feasible combinations      :",
    feasible_options_094
)

print(
    "Infeasible combinations            :",
    tested_options_094 - feasible_options_094
)

print(
    "Users with feasible synchronization:",
    users_with_joint_sync_094
)

print(
    "Users with feasible triple sync    :",
    users_with_joint_triple_094
)

print(
    "Users without feasible sync        :",
    400 - users_with_joint_sync_094
)

print(
    "Approved zero-sync proofs preserved:",
    int(
        zero_proof_audit_09[
            "proof_preserved"
        ].sum()
    ),
    "/400"
)

print(
    "Joint feasibility QA               :",
    f"{len(joint_feasibility_qa_09)}/"
    f"{len(joint_feasibility_qa_09)} PASS"
)

print("Synchronization targets selected   : NO")
print("Daily channel sequences created    : NO")
print("Approved assignments modified      : NO")
print("Original files modified            : NO")
print("Files exported                     : NO")

print("=" * 65)
print("SECTION 9.4 — PASS")


STEP 1 — Approved evidence verified: PASS
STEP 2 — Daily availability prepared: PASS
STEP 3 — Switching-level masks prepared: PASS
STEP 4 — Exact joint DP prepared: PASS
STEP 5 — Joint feasibility evaluated: PASS
Evaluated combinations: 5,624
STEP 6 — User-level feasibility profile: PASS

JOINT FEASIBILITY BY CHANNEL


,channel_configuration,attackers,users_with_feasible_sync,users_with_feasible_triple
0,E+U,117,112,0
1,E+H,117,109,0
2,U+H,27,26,0
3,E+U+H,139,136,83



JOINT FEASIBILITY BY SWITCHING


,switching_level,attackers,users_with_feasible_sync,users_with_feasible_triple
0,Low,133,125,12
1,Medium,134,131,31
2,High,133,127,40



JOINT FEASIBILITY BY DURATION × CHANNEL


,window_workdays,channel_configuration,attackers,users_with_feasible_sync,users_with_feasible_triple,total_feasible_combinations
0,5,E+H,7,2,0,9
1,5,E+U,16,13,0,47
2,5,E+U+H,10,7,1,25
3,5,U+H,17,17,0,37
4,10,E+H,24,21,0,58
5,10,E+U,15,13,0,66
6,10,E+U+H,24,24,10,135
7,10,U+H,7,6,0,19
8,20,E+H,23,23,0,120
9,20,E+U,13,13,0,123


STEP 7 — Joint feasibility summarized: PASS

SECTION 9.4 QUALITY ASSURANCE


,check,status
0,400 approved attackers evaluated,PASS
1,Unique synchronization-count keys,PASS
2,Every attacker has a feasible option,PASS
3,Zero synchronization feasible for all,PASS
4,Approved level-filtered proof preserved,PASS
5,Feasible options retain switching evidence,PASS
6,Three-channel synchronization only in E+U+H,PASS
7,Feasible synchronization within natural opport...,PASS
8,Feasible triple synchronization within natural...,PASS
9,Scenario summaries reconcile,PASS


STEP 8 — Joint feasibility QA: PASS

NOTEBOOK 09 — SECTION 9.4 SUMMARY
Approved attackers                 : 400
Tested count combinations          : 5624
Jointly feasible combinations      : 5260
Infeasible combinations            : 364
Users with feasible synchronization: 383
Users with feasible triple sync    : 83
Users without feasible sync        : 17
Approved zero-sync proofs preserved: 400 /400
Joint feasibility QA               : 10/10 PASS
Synchronization targets selected   : NO
Daily channel sequences created    : NO
Approved assignments modified      : NO
Original files modified            : NO
Files exported                     : NO
SECTION 9.4 — PASS



## 9.5.1 — Feasibility-Guided Multichannel-Day Assignment

### Objective

Assign synchronization targets to the 400 approved attackers using Table B.3, while allowing variation among jointly feasible synchronization configurations.

The selection uses the validated joint feasibility results from Section 9.4 without modifying previously approved attacker assignments.

### Approved Multichannel-Day Rules

| Attack Duration | Two-Channel Days | Three-Channel Days |
|---|---|---|
| 5 workdays | 0 | 0 |
| 10 workdays | 1 | 0 |
| 20 workdays | 2 | 0 |
| 30 workdays | 3 | 0 |
| 40 workdays | 3 or 4 | 1 if feasible |
| 50 workdays | 4 or 5 | 1 or 2 if feasible |

### Selection Priority

For each attacker:

1. Identify the complete Table B.3 configurations that are jointly feasible.
2. If at least one complete configuration is feasible, select one.
3. Otherwise, for 40- and 50-workday scenarios, consider the approved two-channel-only alternatives.
4. If no permitted nonzero configuration is feasible, select zero synchronization.
5. Five-workday scenarios always receive zero synchronization.

### Selection Diversity

When multiple configurations are feasible within the same priority category, select among them using a deterministic, user-specific SHA-256-based choice.

This avoids systematically favoring the minimum synchronization count.

The selection does not impose artificial quotas or guarantee equal representation. Actual diversity depends on the feasible alternatives available to each attacker.

### Feasibility and Integrity

Every selected configuration must:

- Exist in the Section 9.4 joint feasibility results.
- Preserve the approved Switching level.
- Respect naturally available Email, USB, and HTTP channels.
- Preserve approved attack duration, scenario identity, and forced off-days.
- Never introduce synchronization on forced off-days.

### Outputs

- `selected_sync_targets_09`: One selected synchronization configuration per attacker.
- `synchronization_configuration_table_09`: Readable distribution of selected configurations by duration.
- `synchronization_rule_summary_09`: Summary of assignments and exceptions.
- `synchronization_exceptions_09`: Documented fallback assignments.
- `synchronization_selection_qa_09`: Validation checks.

### Scope

Only synchronization counts are selected here. Exact dates, primary channels, and realized daily switching will be constructed in Section 9.5.2.

No source files are modified, and no outputs are exported.


In [14]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.5.1
# FEASIBILITY-GUIDED MULTICHANNEL-DAY ASSIGNMENT
#
# Table B.3 | Deterministic selection diversity
# No DP recomputation | No scheduling | No exports
# ============================================================

import hashlib
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Approved inputs
# ------------------------------------------------------------

if not joint_feasibility_qa_09["status"].eq("PASS").all():
    raise RuntimeError("Section 9.4 QA must pass.")

assignments_0951 = approved_assignments_09.copy(deep=True)
profile_0951 = synchronization_user_profile_09.copy(deep=True)
joint_options_0951 = same_day_joint_feasibility_09.copy(deep=True)

for frame in (
    assignments_0951,
    profile_0951,
    joint_options_0951,
):
    frame["user"] = frame["user"].astype("string")

if (
    len(assignments_0951) != 400
    or not assignments_0951["user"].is_unique
    or not profile_0951["user"].is_unique
):
    raise RuntimeError("Approved input integrity failed.")

if joint_options_0951.duplicated(
    ["user", "target_2ch_days", "target_3ch_days"]
).any():
    raise RuntimeError("Duplicate feasibility options.")


# ------------------------------------------------------------
# STEP 2 — Approved synchronization configurations
# ------------------------------------------------------------

MULTICHANNEL_RULES_0951 = {
    5:  [(0, 0)],
    10: [(1, 0)],
    20: [(2, 0)],
    30: [(3, 0)],
    40: [(3, 1), (4, 1)],
    50: [(4, 1), (4, 2), (5, 1), (5, 2)],
}

TWO_ONLY_FALLBACK_0951 = {
    5:  [],
    10: [],
    20: [],
    30: [],
    40: [(3, 0), (4, 0)],
    50: [(4, 0), (5, 0)],
}

assert set(MULTICHANNEL_RULES_0951) == set(
    DURATION_TARGETS_09
)


# ------------------------------------------------------------
# STEP 3 — Existing jointly feasible options
# ------------------------------------------------------------

feasible_0951 = joint_options_0951.loc[
    joint_options_0951["joint_feasible"]
].copy()

options_by_user_0951 = {
    str(user): group.set_index(
        ["target_2ch_days", "target_3ch_days"],
        drop=False,
    )
    for user, group in feasible_0951.groupby("user")
}

profile_lookup_0951 = profile_0951.set_index("user")

if (
    set(assignments_0951["user"])
    != set(options_by_user_0951)
    or set(assignments_0951["user"])
    != set(profile_lookup_0951.index)
):
    raise RuntimeError("User coverage mismatch.")


# ------------------------------------------------------------
# STEP 4 — Deterministic selection among feasible options
#
# SHA-256 supplies a reproducible user-specific choice.
# No automatic minimum-count preference.
# ------------------------------------------------------------

SELECTION_SEED_0951 = 20261118

def deterministic_choice_0951(user, pairs, category):

    pairs = sorted(set(pairs))

    if not pairs:
        raise ValueError("Cannot choose from empty options.")

    if len(pairs) == 1:
        return pairs[0]

    key = (
        f"{SELECTION_SEED_0951}|"
        f"{user}|{category}"
    ).encode("utf-8")

    digest = hashlib.sha256(key).digest()

    selected_index = int.from_bytes(
        digest,
        byteorder="big",
    ) % len(pairs)

    return pairs[selected_index]


selection_rows_0951 = []

for row in assignments_0951.itertuples(index=False):

    user = str(row.user)
    duration = int(row.window_workdays)

    options = options_by_user_0951[user]
    profile = profile_lookup_0951.loc[user]

    feasible_pairs = set(options.index.tolist())

    full_matches = [
        pair
        for pair in MULTICHANNEL_RULES_0951[duration]
        if pair in feasible_pairs
    ]

    two_only_matches = [
        pair
        for pair in TWO_ONLY_FALLBACK_0951[duration]
        if pair in feasible_pairs
    ]

    # Priority 1 — Complete synchronization rule
    if full_matches:

        eligible_pairs = full_matches

        chosen_pair = deterministic_choice_0951(
            user, eligible_pairs, "FULL_RULE"
        )

        rule_status = (
            "RULE_ZERO"
            if duration == 5
            else "FULL_RULE"
        )

        fallback_reason = "NONE"

    # Priority 2 — Two-channel-only alternative
    elif two_only_matches:

        eligible_pairs = two_only_matches

        chosen_pair = deterministic_choice_0951(
            user, eligible_pairs, "TWO_ONLY"
        )

        rule_status = "TWO_ONLY_FALLBACK"
        fallback_reason = "FULL_RULE_NOT_FEASIBLE"

    # Priority 3 — Zero synchronization
    else:

        eligible_pairs = [(0, 0)]
        chosen_pair = (0, 0)

        rule_status = "ZERO_FALLBACK"

        natural_two_plus = int(
            profile["two_plus_days"]
        )

        natural_three = int(
            profile["three_channel_days"]
        )

        permitted_nonzero = [
            pair
            for pair in (
                MULTICHANNEL_RULES_0951[duration]
                + TWO_ONLY_FALLBACK_0951[duration]
            )
            if pair != (0, 0)
        ]

        capacity_possible = any(
            two + three <= natural_two_plus
            and three <= natural_three
            for two, three in permitted_nonzero
        )

        fallback_reason = (
            "JOINT_FEASIBILITY_LIMIT"
            if capacity_possible
            else "NATURAL_OPPORTUNITY_LIMIT"
        )

    if chosen_pair not in feasible_pairs:
        raise RuntimeError(
            f"Selected option is infeasible: {user}"
        )

    chosen = options.loc[chosen_pair]

    two_days, three_days = map(int, chosen_pair)
    active_days = int(row.proposed_active_days)

    selection_rows_0951.append({
        "user": user,
        "scenario_option_id": row.scenario_option_id,
        "window_workdays": duration,
        "channel_configuration": str(
            row.channel_configuration
        ),
        "switching_level": str(
            row.switching_level
        ),
        "stealth_level": str(
            row.stealth_level
        ),
        "active_days": active_days,
        "forced_offdays": int(
            row.forced_offdays
        ),
        "target_2ch_days": two_days,
        "target_3ch_days": three_days,
        "total_sync_days": two_days + three_days,
        "single_channel_days": (
            active_days - two_days - three_days
        ),
        "attainable_switch_mask": int(
            chosen["attainable_switch_mask"]
        ),
        "natural_two_plus_days": int(
            profile["two_plus_days"]
        ),
        "natural_three_days": int(
            profile["three_channel_days"]
        ),
        "rule_status": rule_status,
        "fallback_reason": fallback_reason,
        "eligible_option_count": len(eligible_pairs),
        "full_feasible_option_count": len(full_matches),
        "two_only_feasible_option_count": len(
            two_only_matches
        ),
        "selection_seed": SELECTION_SEED_0951,
    })

selection_candidate_0951 = (
    pd.DataFrame(selection_rows_0951)
    .sort_values("user")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 5 — Selection classification
# ------------------------------------------------------------

def classify_pattern_0951(two, three):

    if two == 0 and three == 0:
        return "ZERO"

    if three == 0:
        return "TWO_CHANNEL_ONLY"

    if two == 0:
        return "THREE_CHANNEL_ONLY"

    return "MIXED_TWO_AND_THREE"


selection_candidate_0951["selected_pattern"] = [
    classify_pattern_0951(two, three)
    for two, three in zip(
        selection_candidate_0951["target_2ch_days"],
        selection_candidate_0951["target_3ch_days"],
    )
]

selection_candidate_0951["has_sync"] = (
    selection_candidate_0951["total_sync_days"] > 0
)

selection_candidate_0951["has_triple"] = (
    selection_candidate_0951["target_3ch_days"] > 0
)

selection_candidate_0951["selection_method"] = (
    "TABLE_B3_DETERMINISTIC_FEASIBLE_DIVERSITY"
)

selection_candidate_0951["selection_status"] = (
    "PROVISIONAL"
)


# ------------------------------------------------------------
# STEP 6 — Summary and fallback details
# ------------------------------------------------------------

synchronization_rule_summary_09 = (
    selection_candidate_0951
    .groupby("window_workdays")
    .agg(
        attackers=("user", "size"),
        full_rule=(
            "rule_status",
            lambda s: int(s.eq("FULL_RULE").sum()),
        ),
        rule_zero=(
            "rule_status",
            lambda s: int(s.eq("RULE_ZERO").sum()),
        ),
        two_only_fallbacks=(
            "rule_status",
            lambda s: int(
                s.eq("TWO_ONLY_FALLBACK").sum()
            ),
        ),
        zero_fallbacks=(
            "rule_status",
            lambda s: int(
                s.eq("ZERO_FALLBACK").sum()
            ),
        ),
        users_with_sync=("has_sync", "sum"),
        users_with_triple=("has_triple", "sum"),
        two_channel_days=("target_2ch_days", "sum"),
        three_channel_days=("target_3ch_days", "sum"),
    )
    .reindex(list(DURATION_TARGETS_09))
    .reset_index()
)

synchronization_rule_summary_09["sync_user_pct"] = (
    100
    * synchronization_rule_summary_09["users_with_sync"]
    / synchronization_rule_summary_09["attackers"]
).round(2)

synchronization_exceptions_09 = (
    selection_candidate_0951.loc[
        selection_candidate_0951[
            "rule_status"
        ].isin([
            "TWO_ONLY_FALLBACK",
            "ZERO_FALLBACK",
        ])
    ]
    .copy()
)


# ------------------------------------------------------------
# STEP 7 — Consolidated QA
# ------------------------------------------------------------

# Compare original and selected identities without
# rejecting equivalent pandas string dtypes.
approved_ids_0951 = assignments_0951[
    ["user", "scenario_option_id"]
].copy()

selected_ids_0951 = selection_candidate_0951[
    ["user", "scenario_option_id"]
].copy()

for frame in (approved_ids_0951, selected_ids_0951):
    frame["user"] = frame["user"].astype("string")
    frame["scenario_option_id"] = (
        frame["scenario_option_id"].astype("string")
    )

approved_ids_0951 = (
    approved_ids_0951.sort_values(
        ["user", "scenario_option_id"]
    ).reset_index(drop=True)
)

selected_ids_0951 = (
    selected_ids_0951.sort_values(
        ["user", "scenario_option_id"]
    ).reset_index(drop=True)
)

identity_match_0951 = approved_ids_0951.equals(
    selected_ids_0951
)

feasible_lookup_0951 = {
    (str(row.user),
     int(row.target_2ch_days),
     int(row.target_3ch_days)):
        int(row.attainable_switch_mask)
    for row in feasible_0951.itertuples(index=False)
}


def validate_selection_0951(row):

    duration = int(row.window_workdays)

    pair = (
        int(row.target_2ch_days),
        int(row.target_3ch_days),
    )

    user = str(row.user)

    feasible_mask = feasible_lookup_0951.get(
        (user, *pair)
    )

    if feasible_mask is None:
        return False

    if int(row.attainable_switch_mask) != feasible_mask:
        return False

    if feasible_mask <= 0:
        return False

    if row.rule_status == "RULE_ZERO":
        return duration == 5 and pair == (0, 0)

    if row.rule_status == "FULL_RULE":
        return (
            duration != 5
            and pair in MULTICHANNEL_RULES_0951[duration]
        )

    if row.rule_status == "TWO_ONLY_FALLBACK":
        return (
            pair in TWO_ONLY_FALLBACK_0951[duration]
            and int(row.full_feasible_option_count) == 0
        )

    if row.rule_status == "ZERO_FALLBACK":
        return (
            duration != 5
            and pair == (0, 0)
            and int(row.full_feasible_option_count) == 0
            and int(row.two_only_feasible_option_count) == 0
        )

    return False


policy_valid_0951 = all(
    validate_selection_0951(row)
    for row in selection_candidate_0951.itertuples(
        index=False
    )
)

# Independently reproduce the deterministic choice.
choice_reproducible_0951 = True

for row in selection_candidate_0951.itertuples(index=False):

    user = str(row.user)
    duration = int(row.window_workdays)

    candidates = options_by_user_0951[user]
    feasible_pairs = set(candidates.index.tolist())

    full = [
        p for p in MULTICHANNEL_RULES_0951[duration]
        if p in feasible_pairs
    ]

    two_only = [
        p for p in TWO_ONLY_FALLBACK_0951[duration]
        if p in feasible_pairs
    ]

    if full:
        expected = deterministic_choice_0951(
            user, full, "FULL_RULE"
        )
    elif two_only:
        expected = deterministic_choice_0951(
            user, two_only, "TWO_ONLY"
        )
    else:
        expected = (0, 0)

    actual = (
        int(row.target_2ch_days),
        int(row.target_3ch_days),
    )

    if actual != expected:
        choice_reproducible_0951 = False
        break


checks_0951 = {
    "400 unique attackers":
        len(selection_candidate_0951) == 400
        and selection_candidate_0951["user"].is_unique,

    "Approved identities preserved":
        identity_match_0951,

    "Approved rules and fallback priority":
        policy_valid_0951,

    "Deterministic selection reproducible":
        choice_reproducible_0951,

    "Five-day synchronization zero":
        selection_candidate_0951.loc[
            selection_candidate_0951[
                "window_workdays"
            ].eq(5),
            "total_sync_days",
        ].eq(0).all(),

    "Natural opportunity bounds":
        selection_candidate_0951[
            "total_sync_days"
        ].le(
            selection_candidate_0951[
                "natural_two_plus_days"
            ]
        ).all()
        and selection_candidate_0951[
            "target_3ch_days"
        ].le(
            selection_candidate_0951[
                "natural_three_days"
            ]
        ).all(),

    "Triple synchronization only in E+U+H":
        selection_candidate_0951.loc[
            selection_candidate_0951["has_triple"],
            "channel_configuration",
        ].eq("E+U+H").all(),

    "Daily counts reconcile":
        (
            selection_candidate_0951["single_channel_days"]
            + selection_candidate_0951["target_2ch_days"]
            + selection_candidate_0951["target_3ch_days"]
        ).eq(
            selection_candidate_0951["active_days"]
        ).all(),

    "All duration totals preserved":
        synchronization_rule_summary_09[
            "attackers"
        ].tolist() == list(
            DURATION_TARGETS_09.values()
        ),

    "Fallback counts reconcile":
        len(synchronization_exceptions_09)
        == int(
            synchronization_rule_summary_09[
                ["two_only_fallbacks", "zero_fallbacks"]
            ].sum().sum()
        ),
}

synchronization_selection_qa_09 = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if bool(ok) else "FAIL",
    }
    for name, ok in checks_0951.items()
])

failed_0951 = synchronization_selection_qa_09.loc[
    synchronization_selection_qa_09["status"].ne("PASS")
]

if not failed_0951.empty:
    display(failed_0951)
    raise RuntimeError(
        "Section 9.5.1 QA failed."
    )

selected_sync_targets_09 = (
    selection_candidate_0951.copy(deep=True)
)


# ------------------------------------------------------------
# STEP 8 — Readable configuration distribution
#
# Each row explains:
# - Attack duration
# - Two-channel days per attacker
# - Three-channel days per attacker
# - Number of attackers with that exact configuration
# ------------------------------------------------------------

synchronization_configuration_table_09 = (
    selected_sync_targets_09
    .groupby(
        [
            "window_workdays",
            "target_2ch_days",
            "target_3ch_days",
            "rule_status",
        ],
        dropna=False,
    )
    .agg(
        attackers=("user", "size"),
    )
    .reset_index()
    .sort_values(
        [
            "window_workdays",
            "target_2ch_days",
            "target_3ch_days",
        ]
    )
    .reset_index(drop=True)
)

status_labels_0951 = {
    "RULE_ZERO": "Zero by rule",
    "FULL_RULE": "Full synchronization",
    "TWO_ONLY_FALLBACK": "Two-channel only",
    "ZERO_FALLBACK": "Zero fallback",
}

readable_table_0951 = (
    synchronization_configuration_table_09
    .rename(
        columns={
            "window_workdays": "Attack duration",
            "target_2ch_days": "Two-channel days",
            "target_3ch_days": "Three-channel days",
            "attackers": "Attackers",
            "rule_status": "Selection type",
        }
    )
    .copy()
)

readable_table_0951["Selection type"] = (
    readable_table_0951["Selection type"]
    .map(status_labels_0951)
)

readable_table_0951["Configuration"] = (
    "("
    + readable_table_0951[
        "Two-channel days"
    ].astype(str)
    + ", "
    + readable_table_0951[
        "Three-channel days"
    ].astype(str)
    + ")"
)

readable_table_0951 = readable_table_0951[
    [
        "Attack duration",
        "Configuration",
        "Two-channel days",
        "Three-channel days",
        "Attackers",
        "Selection type",
    ]
]


print("\nSELECTED SYNCHRONIZATION CONFIGURATIONS")
print("Each row represents one selected configuration.")

display(
    readable_table_0951.reset_index(drop=True)
)



# ------------------------------------------------------------
# STEP 9 — Diversity evidence
# ------------------------------------------------------------

diversity_summary_0951 = (
    selected_sync_targets_09
    .groupby("window_workdays")
    .agg(
        attackers=("user", "size"),
        users_with_multiple_options=(
            "eligible_option_count",
            lambda s: int((s > 1).sum()),
        ),
        users_with_sync=("has_sync", "sum"),
        users_with_triple=("has_triple", "sum"),
    )
    .reset_index()
)

distinct_0951 = (
    synchronization_configuration_table_09
    .groupby("window_workdays")
    .size()
    .rename("configurations_used")
    .reset_index()
)

diversity_summary_0951 = (
    diversity_summary_0951.merge(
        distinct_0951,
        on="window_workdays",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "window_workdays": "Duration",
            "attackers": "Attackers",
            "users_with_multiple_options": "Multiple feasible options",
            "users_with_sync": "With synchronization",
            "users_with_triple": "With triple synchronization",
            "configurations_used": "Distinct configurations",
        }
    )
)


print("\nSELECTION DIVERSITY BY DURATION")

display(
    diversity_summary_0951.reset_index(drop=True)
)



# ------------------------------------------------------------
# STEP 10 — Concise final summary
# ------------------------------------------------------------

print("\n" + "=" * 56)
print("SECTION 9.5.1 — FINAL SUMMARY")
print("=" * 56)
print("Approved attackers       : 400")

print(
    "Users with synchronization:",
    int(selected_sync_targets_09["has_sync"].sum())
)

print(
    "Users with triple sync   :",
    int(selected_sync_targets_09["has_triple"].sum())
)

print(
    "Two-channel sync days    :",
    int(selected_sync_targets_09["target_2ch_days"].sum())
)

print(
    "Three-channel sync days  :",
    int(selected_sync_targets_09["target_3ch_days"].sum())
)

print(
    "Two-channel fallbacks    :",
    int(
        selected_sync_targets_09["rule_status"]
        .eq("TWO_ONLY_FALLBACK").sum()
    )
)

print(
    "Zero fallbacks           :",
    int(
        selected_sync_targets_09["rule_status"]
        .eq("ZERO_FALLBACK").sum()
    )
)

print(
    "Users with >1 option     :",
    int(
        selected_sync_targets_09[
            "eligible_option_count"
        ].gt(1).sum()
    )
)

print(
    "QA                       :",
    f"{len(synchronization_selection_qa_09)}/"
    f"{len(synchronization_selection_qa_09)} PASS"
)

print("Daily schedules built    : NO")
print("Files exported           : NO")
print("=" * 56)
print("SECTION 9.5.1 — PASS")



SELECTED SYNCHRONIZATION CONFIGURATIONS
Each row represents one selected configuration.


,Attack duration,Configuration,Two-channel days,Three-channel days,Attackers,Selection type
0,5,"(0, 0)",0,0,50,Zero by rule
1,10,"(0, 0)",0,0,6,Zero fallback
2,10,"(1, 0)",1,0,64,Full synchronization
3,20,"(0, 0)",0,0,2,Zero fallback
4,20,"(2, 0)",2,0,68,Full synchronization
5,30,"(3, 0)",3,0,70,Full synchronization
6,40,"(3, 0)",3,0,21,Two-channel only
7,40,"(3, 1)",3,1,10,Full synchronization
8,40,"(4, 0)",4,0,30,Two-channel only
9,40,"(4, 1)",4,1,9,Full synchronization



SELECTION DIVERSITY BY DURATION


,Duration,Attackers,Multiple feasible options,With synchronization,With triple synchronization,Distinct configurations
0,5,50,0,0,0,1
1,10,70,0,64,0,2
2,20,70,0,68,0,2
3,30,70,0,70,0,1
4,40,70,68,70,19,4
5,50,70,70,70,14,5



SECTION 9.5.1 — FINAL SUMMARY
Approved attackers       : 400
Users with synchronization: 342
Users with triple sync   : 33
Two-channel sync days    : 970
Three-channel sync days  : 37
Two-channel fallbacks    : 107
Zero fallbacks           : 8
Users with >1 option     : 138
QA                       : 10/10 PASS
Daily schedules built    : NO
Files exported           : NO
SECTION 9.5.1 — PASS



## 9.5.2 — Daily Channel Sequence Construction

### Objective

Construct a feasible daily channel schedule for each of the 400 approved attackers using the synchronization targets selected in Section 9.5.1.

The schedule must satisfy synchronization and Primary Channel switching constraints simultaneously, without modifying previously approved attacker assignments.

### Inputs

- `daily_window_09`: Approved daily windows, natural channel availability, and forced off-days.
- `selected_sync_targets_09`: Selected two-channel and three-channel synchronization targets.
- `joint_feasibility_qa_09`: Section 9.4 validation evidence.

### Construction Method

A memoized dynamic programming procedure reconstructs a feasible channel sequence for every attacker.

For each active workday:

1. Select one naturally available, approved Primary Channel.
2. Select a scheduled channel set containing that Primary Channel.
3. Use exactly one, two, or three distinct channels according to the selected daily action.
4. Preserve the required total number of two-channel and three-channel synchronization days.
5. Count a channel switch whenever the Primary Channel differs from the previous active day's Primary Channel.
6. Require every channel in the attacker's approved configuration to appear as a Primary Channel at least once.

Forced off-days receive no scheduled channels and do not count as active days or switching transitions.

### Switching Reconstruction

The selected joint feasibility bitmask identifies the attainable switching counts compatible with the attacker's chosen synchronization configuration and approved Switching level.

A deterministic target is selected from these attainable counts.

Backtracking recovers a daily sequence satisfying that exact count.

### Scheduling Preference

Among feasible daily actions, prefer schedules that distribute synchronization days and Primary Channel switches across the attack window.

This is a deterministic scheduling preference, not an additional feasibility requirement.

### Validation

Verify that:

- All 400 approved attackers are preserved.
- Every approved workday appears exactly once.
- Active and forced off-day counts remain unchanged.
- Every scheduled channel is naturally available and approved.
- Every active day has exactly one Primary Channel.
- Scheduled two-channel and three-channel days match Section 9.5.1 exactly.
- Realized switching counts match the selected attainable counts.
- Every approved channel appears as a Primary Channel.
- Forced off-days contain no scheduled attack channels.
- The reconstructed sequence respects the approved scenario and duration.

### Outputs

- `daily_attack_schedule_09`: Daily planned Primary Channels and synchronized channel sets.
- `daily_sequence_user_summary_09`: Per-attacker schedule and switching validation.
- `daily_sequence_duration_summary_09`: Aggregate results by attack duration.
- `daily_sequence_qa_09`: Construction integrity checks.

### Scope

This section produces a validated dry-run schedule only.

It does not create malicious events, alter event counts, inject data, modify the source dataset, or export files.


In [15]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.5.2
# DAILY CHANNEL SEQUENCE CONSTRUCTION
#
# Exact feasibility-guided reconstruction
# Memoized DP + deterministic backtracking
# No injection | No source changes | No exports
# ============================================================

from functools import lru_cache
import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Approved input verification
# ------------------------------------------------------------

if not synchronization_selection_qa_09["status"].eq("PASS").all():
    raise RuntimeError("Section 9.5.1 QA must pass.")

if not joint_feasibility_qa_09["status"].eq("PASS").all():
    raise RuntimeError("Section 9.4 QA must pass.")

targets_0952 = selected_sync_targets_09.copy(deep=True)
daily_0952 = daily_window_09.copy(deep=True)

targets_0952["user"] = targets_0952["user"].astype("string")
daily_0952["user"] = daily_0952["user"].astype("string")

if len(targets_0952) != 400 or not targets_0952["user"].is_unique:
    raise RuntimeError("Invalid approved synchronization targets.")

if not targets_0952["selection_status"].eq("PROVISIONAL").all():
    raise RuntimeError("Unexpected selection status.")

if daily_0952.duplicated(["user", "workday_index"]).any():
    raise RuntimeError("Duplicate daily window keys.")

if set(targets_0952["user"]) != set(daily_0952["user"]):
    raise RuntimeError("Target and daily-window users differ.")

if not daily_0952["is_active_day"].eq(
    ~daily_0952["is_forced_offday"].astype(bool)
).all():
    raise RuntimeError("Inconsistent active/off-day flags.")

# Fixed channel encoding from Section 9.2.
CHANNEL_BITS_0952 = (1, 2, 4)

CHANNEL_NAMES_0952 = {
    1: "E",
    2: "U",
    4: "H",
}

MASK_NAMES_0952 = {
    0: "NONE",
    1: "E",
    2: "U",
    3: "E+U",
    4: "H",
    5: "E+H",
    6: "U+H",
    7: "E+U+H",
}

CONFIG_MASKS_0952 = {
    "E+U": 3,
    "E+H": 5,
    "U+H": 6,
    "E+U+H": 7,
}

daily_0952 = daily_0952.sort_values(
    ["user", "workday_index"]
).reset_index(drop=True)


# ------------------------------------------------------------
# STEP 2 — Enumerate permitted daily actions
#
# Each action:
# (primary_channel_mask, scheduled_channel_mask,
#  two_channel_increment, three_channel_increment)
# ------------------------------------------------------------

@lru_cache(maxsize=None)
def daily_actions_0952(available_mask):

    available_mask = int(available_mask)

    primaries = [
        bit for bit in CHANNEL_BITS_0952
        if available_mask & bit
    ]

    actions = []

    for primary in primaries:

        for channel_mask in range(1, 8):

            if channel_mask & ~available_mask:
                continue

            if not channel_mask & primary:
                continue

            channel_count = channel_mask.bit_count()

            actions.append((
                primary,
                channel_mask,
                int(channel_count == 2),
                int(channel_count == 3),
            ))

    return tuple(actions)


# ------------------------------------------------------------
# STEP 3 — Reconstruct one exact feasible sequence
#
# DP result:
# A bitset of attainable remaining switch counts.
#
# State:
# (day_index, used_primary_mask, previous_primary,
#  remaining_two_days, remaining_three_days)
#
# Switching is counted across consecutive ACTIVE days.
# ------------------------------------------------------------

def reconstruct_user_0952(
    available_masks,
    approved_mask,
    target_two,
    target_three,
    approved_switch_mask,
):

    masks = tuple(
        int(mask) for mask in available_masks
    )

    n = len(masks)

    if n == 0:
        raise RuntimeError("Attacker has no active days.")

    if any(mask == 0 for mask in masks):
        raise RuntimeError(
            "Active day without an approved channel."
        )

    target_two = int(target_two)
    target_three = int(target_three)
    approved_mask = int(approved_mask)
    approved_switch_mask = int(approved_switch_mask)

    # Suffix capacities provide safe pruning.
    suffix_multi = [0] * (n + 1)
    suffix_triple = [0] * (n + 1)
    suffix_union = [0] * (n + 1)

    for i in range(n - 1, -1, -1):

        count = masks[i].bit_count()

        suffix_multi[i] = (
            suffix_multi[i + 1] + int(count >= 2)
        )

        suffix_triple[i] = (
            suffix_triple[i + 1] + int(count == 3)
        )

        suffix_union[i] = (
            suffix_union[i + 1] | masks[i]
        )

    @lru_cache(maxsize=None)
    def possible(
        i, used_mask, previous_primary,
        remaining_two, remaining_three,
    ):

        if remaining_two < 0 or remaining_three < 0:
            return 0

        if (
            remaining_two + remaining_three
            > suffix_multi[i]
        ):
            return 0

        if remaining_three > suffix_triple[i]:
            return 0

        if (
            (used_mask | suffix_union[i])
            & approved_mask
        ) != approved_mask:
            return 0

        if i == n:

            if (
                remaining_two == 0
                and remaining_three == 0
                and used_mask == approved_mask
            ):
                return 1  # Zero remaining switches.

            return 0

        attainable = 0

        for primary, channel_mask, add_two, add_three in (
            daily_actions_0952(masks[i])
        ):

            if add_two > remaining_two:
                continue

            if add_three > remaining_three:
                continue

            switch = int(
                previous_primary != 0
                and primary != previous_primary
            )

            suffix_bits = possible(
                i + 1,
                used_mask | primary,
                primary,
                remaining_two - add_two,
                remaining_three - add_three,
            )

            if suffix_bits:
                attainable |= suffix_bits << switch

        return attainable

    root_mask = possible(
        0, 0, 0, target_two, target_three
    )

    if root_mask == 0:
        raise RuntimeError(
            "No daily sequence satisfies the selected targets."
        )

    # Independent reconstruction must preserve
    # all switch counts advertised as feasible in 9.4.
    if (
        root_mask & approved_switch_mask
    ) != approved_switch_mask:
        raise RuntimeError(
            "Section 9.4 switching evidence "
            "does not match reconstructed feasibility."
        )

    attainable_counts = [
        switches
        for switches in range(n)
        if (approved_switch_mask >> switches) & 1
        and (root_mask >> switches) & 1
    ]

    if not attainable_counts:
        raise RuntimeError(
            "No approved switching count is attainable."
        )

    # Deterministic middle feasible switching target.
    target_switches = attainable_counts[
        (len(attainable_counts) - 1) // 2
    ]

    # --------------------------------------------------------
    # Backtracking with temporal-spreading preference
    # --------------------------------------------------------

    remaining_two = target_two
    remaining_three = target_three
    remaining_switches = target_switches

    used_mask = 0
    previous_primary = 0

    reconstructed = []

    for i, available_mask in enumerate(masks):

        used_two = target_two - remaining_two
        used_three = target_three - remaining_three
        used_switches = target_switches - remaining_switches

        fraction = (i + 1) / n

        candidates = []

        for primary, channel_mask, add_two, add_three in (
            daily_actions_0952(available_mask)
        ):

            if (
                add_two > remaining_two
                or add_three > remaining_three
            ):
                continue

            switch = int(
                previous_primary != 0
                and primary != previous_primary
            )

            if switch > remaining_switches:
                continue

            next_bits = possible(
                i + 1,
                used_mask | primary,
                primary,
                remaining_two - add_two,
                remaining_three - add_three,
            )

            required_suffix = (
                remaining_switches - switch
            )

            if not (
                (next_bits >> required_suffix) & 1
            ):
                continue

            # Prefer evenly spread sync/switch counts,
            # without excluding any feasible solution.
            score = (
                abs(
                    (used_two + add_two)
                    - target_two * fraction
                )
                +
                abs(
                    (used_three + add_three)
                    - target_three * fraction
                )
                +
                0.4 * abs(
                    (used_switches + switch)
                    - target_switches * fraction
                )
            )

            candidates.append((
                score,
                primary,
                channel_mask,
                add_two,
                add_three,
                switch,
            ))

        if not candidates:
            raise RuntimeError(
                f"Backtracking failed at active day {i + 1}."
            )

        candidates.sort(
            key=lambda item: (
                item[0],
                item[1],
                item[2],
            )
        )

        (
            _,
            primary,
            channel_mask,
            add_two,
            add_three,
            switch,
        ) = candidates[0]

        reconstructed.append({
            "active_day_index": i + 1,
            "primary_channel_mask": primary,
            "scheduled_channel_mask": channel_mask,
            "is_switch": bool(switch),
        })

        used_mask |= primary
        previous_primary = primary

        remaining_two -= add_two
        remaining_three -= add_three
        remaining_switches -= switch

    if not (
        remaining_two == 0
        and remaining_three == 0
        and remaining_switches == 0
        and used_mask == approved_mask
    ):
        raise RuntimeError(
            "Reconstructed sequence failed final reconciliation."
        )

    return reconstructed, target_switches


# ------------------------------------------------------------
# STEP 4 — Construct daily schedules for 400 attackers
# ------------------------------------------------------------

daily_groups_0952 = {
    str(user): group.copy()
    for user, group in daily_0952.groupby(
        "user", sort=False
    )
}

action_rows_0952 = []
user_rows_0952 = []

for target in targets_0952.itertuples(index=False):

    user = str(target.user)

    group = daily_groups_0952[user]

    active = group.loc[
        group["is_active_day"].astype(bool)
    ].sort_values("workday_index")

    active_masks = (
        active["available_channel_mask"]
        .astype(int)
        .tolist()
    )

    configuration = str(
        target.channel_configuration
    )

    approved_mask = CONFIG_MASKS_0952[configuration]

    # Validate immutable daily boundaries.
    if len(group) != int(target.window_workdays):
        raise RuntimeError(
            f"Window duration mismatch: {user}"
        )

    if len(active) != int(target.active_days):
        raise RuntimeError(
            f"Active-day count mismatch: {user}"
        )

    if (
        int(group["is_forced_offday"].sum())
        != int(target.forced_offdays)
    ):
        raise RuntimeError(
            f"Forced off-day count mismatch: {user}"
        )

    try:
        sequence, target_switches = reconstruct_user_0952(
            available_masks=active_masks,
            approved_mask=approved_mask,
            target_two=int(target.target_2ch_days),
            target_three=int(target.target_3ch_days),
            approved_switch_mask=int(
                target.attainable_switch_mask
            ),
        )
    except RuntimeError as exc:
        raise RuntimeError(
            f"User {user}: {exc}"
        ) from exc

    for (_, day_row), action in zip(
        active.iterrows(), sequence
    ):

        action_rows_0952.append({
            "user": user,
            "workday_index": int(
                day_row["workday_index"]
            ),
            **action,
        })

    user_rows_0952.append({
        "user": user,
        "scenario_option_id": target.scenario_option_id,
        "window_workdays": int(
            target.window_workdays
        ),
        "channel_configuration": configuration,
        "switching_level": str(
            target.switching_level
        ),
        "target_2ch_days": int(
            target.target_2ch_days
        ),
        "target_3ch_days": int(
            target.target_3ch_days
        ),
        "target_switches": int(target_switches),
        "active_days": int(target.active_days),
        "forced_offdays": int(
            target.forced_offdays
        ),
    })


# ------------------------------------------------------------
# STEP 5 — Attach reconstructed actions to daily windows
# ------------------------------------------------------------

actions_0952 = pd.DataFrame(action_rows_0952)
user_schedule_targets_0952 = pd.DataFrame(user_rows_0952)

daily_candidate_0952 = daily_0952.merge(
    actions_0952,
    on=["user", "workday_index"],
    how="left",
    validate="one_to_one",
    indicator="_schedule_merge",
)

is_active_0952 = daily_candidate_0952[
    "is_active_day"
].astype(bool)

if not daily_candidate_0952.loc[
    is_active_0952, "_schedule_merge"
].eq("both").all():
    raise RuntimeError("Missing reconstructed active days.")

if not daily_candidate_0952.loc[
    ~is_active_0952, "_schedule_merge"
].eq("left_only").all():
    raise RuntimeError(
        "Unexpected actions on forced off-days."
    )

for column in [
    "primary_channel_mask",
    "scheduled_channel_mask",
    "active_day_index",
]:
    daily_candidate_0952[column] = (
        daily_candidate_0952[column]
        .fillna(0)
        .astype(int)
    )

daily_candidate_0952["is_switch"] = (
    daily_candidate_0952["is_switch"]
    .fillna(False)
    .astype(bool)
)

daily_candidate_0952["primary_channel"] = (
    daily_candidate_0952["primary_channel_mask"]
    .map(MASK_NAMES_0952)
)

daily_candidate_0952["scheduled_channel_set"] = (
    daily_candidate_0952["scheduled_channel_mask"]
    .map(MASK_NAMES_0952)
)

daily_candidate_0952["scheduled_channel_count"] = (
    daily_candidate_0952["scheduled_channel_mask"]
    .map(lambda mask: int(mask).bit_count())
)

daily_candidate_0952["sync_type"] = (
    daily_candidate_0952["scheduled_channel_count"]
    .map({
        0: "FORCED_OFFDAY",
        1: "SINGLE",
        2: "TWO_CHANNEL",
        3: "THREE_CHANNEL",
    })
)

daily_candidate_0952 = (
    daily_candidate_0952
    .drop(columns="_schedule_merge")
    .sort_values(["user", "workday_index"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 6 — Independent realized-switch calculation
#
# Recompute from chronological Primary Channels.
# Forced off-days are skipped, not treated as channels.
# ------------------------------------------------------------

active_sequence_0952 = daily_candidate_0952.loc[
    daily_candidate_0952["is_active_day"]
].copy()

previous_primary_0952 = (
    active_sequence_0952
    .groupby("user")["primary_channel_mask"]
    .shift()
)

active_sequence_0952["recomputed_switch"] = (
    previous_primary_0952.notna()
    &
    active_sequence_0952["primary_channel_mask"]
    .ne(previous_primary_0952)
)

switch_consistent_0952 = bool(
    active_sequence_0952["is_switch"].eq(
        active_sequence_0952["recomputed_switch"]
    ).all()
)

realized_user_0952 = (
    active_sequence_0952
    .groupby("user")
    .agg(
        realized_active_days=("day", "size"),
        realized_two_days=(
            "sync_type",
            lambda s: int(s.eq("TWO_CHANNEL").sum()),
        ),
        realized_three_days=(
            "sync_type",
            lambda s: int(s.eq("THREE_CHANNEL").sum()),
        ),
        realized_switches=(
            "recomputed_switch",
            "sum",
        ),
    )
    .reset_index()
)

primary_coverage_0952 = (
    active_sequence_0952
    .groupby("user")["primary_channel_mask"]
    .agg(
        lambda s: int(
            np.bitwise_or.reduce(
                s.to_numpy(dtype=np.int64)
            )
        )
    )
    .rename("realized_primary_mask")
    .reset_index()
)

realized_user_0952 = realized_user_0952.merge(
    primary_coverage_0952,
    on="user",
    validate="one_to_one",
)

daily_sequence_user_summary_09 = (
    user_schedule_targets_0952
    .merge(
        realized_user_0952,
        on="user",
        validate="one_to_one",
    )
)

daily_sequence_user_summary_09["approved_channel_mask"] = (
    daily_sequence_user_summary_09[
        "channel_configuration"
    ].map(CONFIG_MASKS_0952)
)


# ------------------------------------------------------------
# STEP 7 — Independent QA
# ------------------------------------------------------------

approved_identity_0952 = (
    targets_0952[
        ["user", "scenario_option_id"]
    ].copy()
)

realized_identity_0952 = (
    daily_sequence_user_summary_09[
        ["user", "scenario_option_id"]
    ].copy()
)

for frame in (
    approved_identity_0952,
    realized_identity_0952,
):
    frame["user"] = frame["user"].astype("string")
    frame["scenario_option_id"] = (
        frame["scenario_option_id"].astype("string")
    )

identity_valid_0952 = (
    approved_identity_0952
    .sort_values("user")
    .reset_index(drop=True)
    .equals(
        realized_identity_0952
        .sort_values("user")
        .reset_index(drop=True)
    )
)

daily_keys_valid_0952 = bool(
    not daily_candidate_0952.duplicated(
        ["user", "workday_index"]
    ).any()
    and
    len(daily_candidate_0952) == len(daily_0952)
)

active_counts_valid_0952 = bool(
    daily_sequence_user_summary_09[
        "realized_active_days"
    ].eq(
        daily_sequence_user_summary_09["active_days"]
    ).all()
)

sync_counts_valid_0952 = bool(
    daily_sequence_user_summary_09[
        "realized_two_days"
    ].eq(
        daily_sequence_user_summary_09["target_2ch_days"]
    ).all()
    and
    daily_sequence_user_summary_09[
        "realized_three_days"
    ].eq(
        daily_sequence_user_summary_09["target_3ch_days"]
    ).all()
)

switch_counts_valid_0952 = bool(
    daily_sequence_user_summary_09[
        "realized_switches"
    ].eq(
        daily_sequence_user_summary_09["target_switches"]
    ).all()
)

primary_coverage_valid_0952 = bool(
    daily_sequence_user_summary_09[
        "realized_primary_mask"
    ].eq(
        daily_sequence_user_summary_09[
            "approved_channel_mask"
        ]
    ).all()
)

offday_valid_0952 = bool(
    daily_candidate_0952.loc[
        ~is_active_0952,
        "scheduled_channel_mask",
    ].eq(0).all()
    and
    daily_candidate_0952.loc[
        ~is_active_0952,
        "primary_channel_mask",
    ].eq(0).all()
    and
    ~daily_candidate_0952.loc[
        ~is_active_0952,
        "is_switch",
    ].any()
)

natural_availability_valid_0952 = bool(
    (
        daily_candidate_0952[
            "scheduled_channel_mask"
        ].astype(int)
        &
        ~daily_candidate_0952[
            "available_channel_mask"
        ].astype(int)
    ).eq(0).all()
)

primary_membership_valid_0952 = bool(
    (
        daily_candidate_0952.loc[
            is_active_0952,
            "primary_channel_mask",
        ].astype(int)
        &
        daily_candidate_0952.loc[
            is_active_0952,
            "scheduled_channel_mask",
        ].astype(int)
    ).eq(
        daily_candidate_0952.loc[
            is_active_0952,
            "primary_channel_mask",
        ]
    ).all()
)

active_channels_valid_0952 = bool(
    daily_candidate_0952.loc[
        is_active_0952,
        "scheduled_channel_count",
    ].between(1, 3).all()
    and
    daily_candidate_0952.loc[
        is_active_0952,
        "primary_channel_mask",
    ].isin(CHANNEL_BITS_0952).all()
)

approved_switch_mask_lookup_0952 = (
    targets_0952
    .set_index("user")["attainable_switch_mask"]
    .to_dict()
)

switch_level_valid_0952 = all(
    (
        int(approved_switch_mask_lookup_0952[str(row.user)])
        >> int(row.realized_switches)
    ) & 1
    for row in daily_sequence_user_summary_09.itertuples(
        index=False
    )
)

expected_duration_counts_0952 = {
    int(k): int(v)
    for k, v in DURATION_TARGETS_09.items()
}

duration_valid_0952 = (
    daily_sequence_user_summary_09[
        "window_workdays"
    ].value_counts().to_dict()
    == expected_duration_counts_0952
)

checks_0952 = {
    "Approved identities preserved":
        identity_valid_0952,
    "All approved daily rows preserved":
        daily_keys_valid_0952,
    "Active-day counts preserved":
        active_counts_valid_0952,
    "Synchronization counts realized":
        sync_counts_valid_0952,
    "Switching counts realized":
        switch_counts_valid_0952,
    "Switching independently recomputed":
        switch_consistent_0952,
    "Switching within approved feasible counts":
        switch_level_valid_0952,
    "All approved primary channels represented":
        primary_coverage_valid_0952,
    "Forced off-days contain no attack actions":
        offday_valid_0952,
    "Scheduled channels naturally available":
        natural_availability_valid_0952,
    "Every active primary belongs to its scheduled set":
        primary_membership_valid_0952,
    "Valid active channel cardinalities":
        active_channels_valid_0952,
    "Approved duration distribution preserved":
        duration_valid_0952,
}

daily_sequence_qa_09 = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if bool(ok) else "FAIL",
    }
    for name, ok in checks_0952.items()
])

failed_0952 = daily_sequence_qa_09.loc[
    daily_sequence_qa_09["status"].ne("PASS")
]

if not failed_0952.empty:
    print("\nFAILED QUALITY CHECKS")
    display(failed_0952)
    raise RuntimeError("Section 9.5.2 QA failed.")


# ------------------------------------------------------------
# STEP 8 — Publish validated in-memory results
# ------------------------------------------------------------

daily_attack_schedule_09 = daily_candidate_0952.copy(
    deep=True
)

daily_sequence_duration_summary_09 = (
    daily_sequence_user_summary_09
    .groupby("window_workdays")
    .agg(
        attackers=("user", "size"),
        active_days=("realized_active_days", "sum"),
        two_channel_days=("realized_two_days", "sum"),
        three_channel_days=("realized_three_days", "sum"),
        primary_switches=("realized_switches", "sum"),
        forced_offdays=("forced_offdays", "sum"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# STEP 9 — Essential readable outputs
# ------------------------------------------------------------

print("\nDAILY SEQUENCE RESULTS BY DURATION")

display(
    daily_sequence_duration_summary_09.rename(
        columns={
            "window_workdays": "Duration",
            "attackers": "Attackers",
            "active_days": "Active days",
            "two_channel_days": "Two-channel days",
            "three_channel_days": "Three-channel days",
            "primary_switches": "Primary switches",
            "forced_offdays": "Forced off-days",
        }
    )
)

# Show one attacker with synchronization, if available.
sample_users_0952 = (
    daily_sequence_user_summary_09.loc[
        daily_sequence_user_summary_09[
            "target_3ch_days"
        ].gt(0),
        "user",
    ]
    .astype(str)
    .sort_values()
)

if sample_users_0952.empty:
    sample_users_0952 = (
        daily_sequence_user_summary_09.loc[
            daily_sequence_user_summary_09[
                "target_2ch_days"
            ].gt(0),
            "user",
        ]
        .astype(str)
        .sort_values()
    )

sample_user_0952 = sample_users_0952.iloc[0]

sample_schedule_0952 = (
    daily_attack_schedule_09.loc[
        daily_attack_schedule_09["user"].eq(
            sample_user_0952
        )
    ]
    .sort_values("workday_index")
)

print(f"\nEXAMPLE DAILY SCHEDULE — USER {sample_user_0952}")

display(
    sample_schedule_0952[
        [
            "day",
            "workday_index",
            "day_status",
            "available_channel_set",
            "primary_channel",
            "scheduled_channel_set",
            "sync_type",
            "is_switch",
        ]
    ].head(15).reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 10 — Concise summary
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("SECTION 9.5.2 — DAILY SEQUENCE SUMMARY")
print("=" * 55)

print(
    "Approved attackers      :",
    len(daily_sequence_user_summary_09)
)

print(
    "Scheduled window days   :",
    len(daily_attack_schedule_09)
)

print(
    "Active days             :",
    int(daily_attack_schedule_09["is_active_day"].sum())
)

print(
    "Forced off-days         :",
    int(
        daily_attack_schedule_09[
            "is_forced_offday"
        ].sum()
    )
)

print(
    "Two-channel days        :",
    int(
        daily_sequence_user_summary_09[
            "realized_two_days"
        ].sum()
    )
)

print(
    "Three-channel days      :",
    int(
        daily_sequence_user_summary_09[
            "realized_three_days"
        ].sum()
    )
)

print(
    "Primary switches        :",
    int(
        daily_sequence_user_summary_09[
            "realized_switches"
        ].sum()
    )
)

print(
    "QA                      :",
    f"{len(daily_sequence_qa_09)}/"
    f"{len(daily_sequence_qa_09)} PASS"
)

print("Attack events injected  : NO")
print("Source files modified   : NO")
print("Files exported          : NO")
print("=" * 55)
print("SECTION 9.5.2 — PASS")



DAILY SEQUENCE RESULTS BY DURATION


,Duration,Attackers,Active days,Two-channel days,Three-channel days,Primary switches,Forced off-days
0,5,50,250,0,0,97,0
1,10,70,610,64,0,217,90
2,20,70,1215,136,0,443,185
3,30,70,1775,210,0,622,325
4,40,70,2335,249,19,799,465
5,50,70,2895,311,18,938,605



EXAMPLE DAILY SCHEDULE — USER BBA1230


,day,workday_index,day_status,available_channel_set,primary_channel,scheduled_channel_set,sync_type,is_switch
0,2011-03-11,1,ACTIVE,E,E,E,SINGLE,False
1,2011-03-14,2,ACTIVE,E,E,E,SINGLE,False
2,2011-03-15,3,ACTIVE,E+H,H,H,SINGLE,True
3,2011-03-16,4,ACTIVE,E,E,E,SINGLE,True
4,2011-03-17,5,ACTIVE,E+U+H,E,E+U+H,THREE_CHANNEL,False
5,2011-03-18,6,ACTIVE,E,E,E,SINGLE,False
6,2011-03-21,7,ACTIVE,E+U,U,E+U,TWO_CHANNEL,True
7,2011-03-22,8,ACTIVE,U,U,U,SINGLE,False
8,2011-03-23,9,ACTIVE,H,H,H,SINGLE,True
9,2011-03-24,10,ACTIVE,U,U,U,SINGLE,True



SECTION 9.5.2 — DAILY SEQUENCE SUMMARY
Approved attackers      : 400
Scheduled window days   : 10750
Active days             : 9080
Forced off-days         : 1670
Two-channel days        : 970
Three-channel days      : 37
Primary switches        : 3116
QA                      : 13/13 PASS
Attack events injected  : NO
Source files modified   : NO
Files exported          : NO
SECTION 9.5.2 — PASS



## 9.6 — Independent Schedule Validation

### Objective

Independently validate the daily attack schedule constructed in Section 9.5.2 before approving it for export.

This section does not use the dynamic programming reconstruction algorithm or its intermediate states.

### Validation Sources

- `approved_assignments_09`: Approved attacker identities, scenario configurations, durations, and windows from Notebook 08.
- `selected_sync_targets_09`: Synchronization targets selected in Section 9.5.1.
- `source_user_day_09`: Original processed user-day activity.
- `daily_window_09`: Previously reconstructed attack windows and forced off-day assignments.
- `daily_attack_schedule_09`: Daily schedule requiring independent validation.

### Independent Validation Procedure

**1. Identity and Window Integrity**

Verify all 400 approved attackers, their scenario assignments, window boundaries, workday indices, and daily row counts.

**2. Source-Based Channel Availability**

Recompute naturally active channels directly from the original Email, USB, and HTTP event counts.

Verify that each scheduled channel belongs to both the naturally available channels and the approved channel configuration.

**3. Forced Off-Day Protection**

Verify that forced off-days contain no planned attack channels, Primary Channel, or switching activity.

**4. Synchronization Recalculation**

Recompute the number of scheduled channels per active day and independently count:

- Single-channel days.
- Exactly-two-channel synchronization days.
- Three-channel synchronization days.

Compare these counts with the approved Section 9.5.1 targets.

**5. Primary Channel and Switching Verification**

Recompute Primary Channel switches chronologically between consecutive active days, excluding forced off-days.

Verify that realized switches satisfy the approved Switching level and selected feasible switching counts.

Verify that every approved channel appears as a Primary Channel.

**6. Cross-Source Reconciliation**

Verify that the daily schedule, source user-day records, selected synchronization targets, and approved scenario assignments agree.

### Outputs

- `independent_schedule_user_audit_09`: Independently verified metrics for each attacker.
- `independent_schedule_duration_summary_09`: Recalculated results by attack duration.
- `independent_schedule_qa_09`: Independent validation results.

### Acceptance Criteria

All independent checks must pass before the schedule can be approved for export.

Any discrepancy must raise an explicit error and prevent the independent audit from being marked as approved.

### Scope

This section performs read-only verification.

No attack events are injected, no source data are modified, and no files are exported.


In [17]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.6
# INDEPENDENT DAILY SCHEDULE VALIDATION
#
# Source-based audit | Independent switching recalculation
# No DP | No injection | No exports
# ============================================================

import math
import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Read-only input preparation
# ------------------------------------------------------------

required_qa_096 = [
    synchronization_selection_qa_09,
    daily_sequence_qa_09,
]

if not all(
    qa["status"].eq("PASS").all()
    for qa in required_qa_096
):
    raise RuntimeError(
        "Sections 9.5.1 and 9.5.2 must pass before "
        "independent validation."
    )

schedule_096 = daily_attack_schedule_09.copy(deep=True)
approved_096 = approved_assignments_09.copy(deep=True)
targets_096 = selected_sync_targets_09.copy(deep=True)
windows_096 = daily_window_09.copy(deep=True)

SOURCE_COLUMNS_096 = [
    "user",
    "day",
    "is_weekend",
    "logon_count",
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

source_096 = source_user_day_09[
    SOURCE_COLUMNS_096
].copy()

for frame in (
    schedule_096,
    approved_096,
    targets_096,
    windows_096,
    source_096,
):
    frame["user"] = frame["user"].astype("string")

for frame in (schedule_096, windows_096, source_096):
    frame["day"] = pd.to_datetime(
        frame["day"], errors="raise"
    ).dt.normalize()

CONFIG_MASKS_096 = {
    "E+U": 3,
    "E+H": 5,
    "U+H": 6,
    "E+U+H": 7,
}

CHANNEL_BITS_096 = (1, 2, 4)

# Structural failures must stop validation immediately.
if (
    schedule_096.duplicated(["user", "day"]).any()
    or windows_096.duplicated(["user", "day"]).any()
    or source_096.duplicated(["user", "day"]).any()
):
    raise RuntimeError(
        "Duplicate user-day keys detected."
    )

if (
    not approved_096["user"].is_unique
    or not targets_096["user"].is_unique
):
    raise RuntimeError(
        "Duplicate attacker identifiers detected."
    )


# ------------------------------------------------------------
# STEP 2 — Independent original-source reconstruction
#
# E = Email, U = USB, H = HTTP.
# Reconstruct natural channel activity from event COUNTS.
# No byte or volume information is used.
# ------------------------------------------------------------

source_096 = source_096.loc[
    source_096["user"].isin(
        approved_096["user"]
    )
].copy()

for column in [
    "logon_count",
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]:
    source_096[column] = pd.to_numeric(
        source_096[column],
        errors="raise",
    )

source_096["source_workday_eligible"] = (
    source_096["is_weekend"].eq(0)
    &
    source_096["logon_count"].gt(0)
)

source_096["source_natural_mask"] = (
    source_096["external_email_count"]
    .gt(0).astype(int)
    +
    2 * source_096["usb_copy_count"]
    .gt(0).astype(int)
    +
    4 * source_096["www_upload_count"]
    .gt(0).astype(int)
)

source_096 = source_096[
    [
        "user",
        "day",
        "source_workday_eligible",
        "source_natural_mask",
    ]
]

audit_096 = schedule_096.merge(
    source_096,
    on=["user", "day"],
    how="left",
    validate="many_to_one",
    indicator="_source_match",
)

audit_096 = audit_096.merge(
    approved_096[
        [
            "user",
            "scenario_option_id",
            "window_workdays",
            "window_start",
            "window_end",
            "channel_configuration",
            "switching_level",
            "forced_offdays",
            "proposed_active_days",
        ]
    ].rename(
        columns={
            c: f"approved_{c}"
            for c in [
                "scenario_option_id",
                "window_workdays",
                "window_start",
                "window_end",
                "channel_configuration",
                "switching_level",
                "forced_offdays",
                "proposed_active_days",
            ]
        }
    ),
    on="user",
    how="left",
    validate="many_to_one",
)

audit_096 = audit_096.sort_values(
    ["user", "day"]
).reset_index(drop=True)

audit_096["expected_approved_mask"] = (
    audit_096["approved_channel_configuration"]
    .map(CONFIG_MASKS_096)
)

if audit_096[
    [
        "source_natural_mask",
        "expected_approved_mask",
    ]
].isna().any().any():
    raise RuntimeError(
        "Source records or approved configurations "
        "are missing."
    )

audit_096["expected_available_mask"] = (
    audit_096["source_natural_mask"].astype(int)
    &
    audit_096["expected_approved_mask"].astype(int)
)


# ------------------------------------------------------------
# STEP 3 — Independently reconstruct daily status
# ------------------------------------------------------------

audit_096["audited_forced_offday"] = (
    audit_096["is_forced_offday"].astype(bool)
)

audit_096["audited_active_day"] = (
    ~audit_096["audited_forced_offday"]
)

audit_096["audited_channel_count"] = (
    audit_096["scheduled_channel_mask"]
    .astype(int)
    .map(int.bit_count)
)

audit_096["audited_two_channel_day"] = (
    audit_096["audited_active_day"]
    &
    audit_096["audited_channel_count"].eq(2)
)

audit_096["audited_three_channel_day"] = (
    audit_096["audited_active_day"]
    &
    audit_096["audited_channel_count"].eq(3)
)

audit_096["audited_single_channel_day"] = (
    audit_096["audited_active_day"]
    &
    audit_096["audited_channel_count"].eq(1)
)

audit_096["audited_sync_type"] = np.select(
    [
        audit_096["audited_forced_offday"],
        audit_096["audited_single_channel_day"],
        audit_096["audited_two_channel_day"],
        audit_096["audited_three_channel_day"],
    ],
    [
        "FORCED_OFFDAY",
        "SINGLE",
        "TWO_CHANNEL",
        "THREE_CHANNEL",
    ],
    default="INVALID",
)


# ------------------------------------------------------------
# STEP 4 — Independently recalculate switching
#
# Important:
# Exclude forced off-days before shifting Primary Channel.
# This makes switching relative to the previous ACTIVE day.
# ------------------------------------------------------------

audit_096["audited_switch"] = False

active_096 = audit_096.loc[
    audit_096["audited_active_day"]
].copy()

active_096 = active_096.sort_values(
    ["user", "day"]
)

previous_primary_096 = (
    active_096
    .groupby("user")["primary_channel_mask"]
    .shift()
)

active_096["audited_switch"] = (
    previous_primary_096.notna()
    &
    active_096["primary_channel_mask"]
    .ne(previous_primary_096)
)

audit_096.loc[
    active_096.index, "audited_switch"
] = active_096["audited_switch"].to_numpy(dtype=bool)


# ------------------------------------------------------------
# STEP 5 — Independent per-user aggregation
# ------------------------------------------------------------

def primary_mask_union_096(series):

    result = 0

    for value in series:
        result |= int(value)

    return result


independent_user_candidate_096 = (
    audit_096
    .groupby("user")
    .agg(
        window_rows=("day", "size"),
        first_day=("day", "min"),
        last_day=("day", "max"),
        active_days=("audited_active_day", "sum"),
        forced_offdays=("audited_forced_offday", "sum"),
        single_channel_days=(
            "audited_single_channel_day", "sum"
        ),
        two_channel_days=(
            "audited_two_channel_day", "sum"
        ),
        three_channel_days=(
            "audited_three_channel_day", "sum"
        ),
        switches=("audited_switch", "sum"),
        primary_coverage_mask=(
            "primary_channel_mask",
            primary_mask_union_096,
        ),
    )
    .reset_index()
)

independent_user_candidate_096 = (
    independent_user_candidate_096
    .merge(
        targets_096[
            [
                "user",
                "scenario_option_id",
                "window_workdays",
                "active_days",
                "forced_offdays",
                "target_2ch_days",
                "target_3ch_days",
                "attainable_switch_mask",
            ]
        ].rename(
            columns={
                "scenario_option_id":
                    "target_scenario_option_id",
                "window_workdays":
                    "target_window_workdays",
                "active_days":
                    "target_active_days",
                "forced_offdays":
                    "target_forced_offdays",
            }
        ),
        on="user",
        how="left",
        validate="one_to_one",
        suffixes=("", "_target"),
    )
    .merge(
        approved_096[
            [
                "user",
                "scenario_option_id",
                "window_workdays",
                "channel_configuration",
                "switching_level",
                "window_start",
                "window_end",
                "forced_offdays",
                "proposed_active_days",
            ]
        ],
        on="user",
        how="left",
        validate="one_to_one",
        suffixes=("", "_approved"),
    )
)


# ------------------------------------------------------------
# STEP 6 — Verify approved switching level independently
#
# Five-day scenarios use exact switch counts.
# Longer scenarios use the approved level ranges.
# ------------------------------------------------------------

SWITCH_RANGES_096 = {
    "Low": (0.10, 0.25),
    "Medium": (0.30, 0.45),
    "High": (0.50, 0.65),
}

FIVE_DAY_SWITCHES_096 = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
}


def independently_valid_switches_096(
    duration, active_days, level, switches
):

    duration = int(duration)
    active_days = int(active_days)
    switches = int(switches)
    level = str(level)

    if active_days < 2:
        return False

    if duration == 5:
        return switches == FIVE_DAY_SWITCHES_096[level]

    transitions = active_days - 1

    lower_rate, upper_rate = SWITCH_RANGES_096[level]

    minimum = math.ceil(
        lower_rate * transitions - 1e-10
    )

    maximum = math.floor(
        upper_rate * transitions + 1e-10
    )

    return minimum <= switches <= maximum


independent_user_candidate_096[
    "switching_level_valid"
] = [
    independently_valid_switches_096(
        duration,
        active_days,
        level,
        switches,
    )
    for duration, active_days, level, switches in zip(
        independent_user_candidate_096["window_workdays"],
        independent_user_candidate_096["active_days"],
        independent_user_candidate_096["switching_level"],
        independent_user_candidate_096["switches"],
    )
]

independent_user_candidate_096[
    "attainable_switch_valid"
] = [
    bool((int(mask) >> int(count)) & 1)
    for mask, count in zip(
        independent_user_candidate_096[
            "attainable_switch_mask"
        ],
        independent_user_candidate_096["switches"],
    )
]


# ------------------------------------------------------------
# STEP 7 — Independent integrity checks
# ------------------------------------------------------------

def same_values_096(left, right):
    return (
        left.astype("string")
        .reset_index(drop=True)
        .equals(
            right.astype("string")
            .reset_index(drop=True)
        )
    )


approved_users_096 = set(
    approved_096["user"].astype(str)
)

scheduled_users_096 = set(
    audit_096["user"].astype(str)
)

window_keys_096 = set(
    zip(
        windows_096["user"].astype(str),
        windows_096["day"],
        windows_096["workday_index"].astype(int),
    )
)

schedule_keys_096 = set(
    zip(
        audit_096["user"].astype(str),
        audit_096["day"],
        audit_096["workday_index"].astype(int),
    )
)

# Scenario IDs and immutable assignment fields.
immutable_fields_096 = [
    ("scenario_option_id", "approved_scenario_option_id"),
    ("window_workdays", "approved_window_workdays"),
    ("channel_configuration", "approved_channel_configuration"),
    ("switching_level", "approved_switching_level"),
    ("forced_offdays", "approved_forced_offdays"),
    ("proposed_active_days", "approved_proposed_active_days"),
    ("window_start", "approved_window_start"),
    ("window_end", "approved_window_end"),
]

immutability_valid_096 = True

for left, right in immutable_fields_096:

    if not same_values_096(
        audit_096[left],
        audit_096[right],
    ):
        immutability_valid_096 = False
        break

# Cross-source date boundaries.
approved_starts_096 = pd.to_datetime(
    independent_user_candidate_096["window_start"]
).dt.normalize()

approved_ends_096 = pd.to_datetime(
    independent_user_candidate_096["window_end"]
).dt.normalize()

date_bounds_valid_096 = bool(
    (
        independent_user_candidate_096["first_day"]
        >= approved_starts_096
    ).all()
    and
    (
        independent_user_candidate_096["last_day"]
        <= approved_ends_096
    ).all()
)

# Workday indices must be consecutive within each user.
workday_index_valid_096 = all(
    group["workday_index"].astype(int).tolist()
    == list(range(1, len(group) + 1))
    for _, group in audit_096.groupby(
        "user", sort=False
    )
)

# Source-based channel checks.
natural_mask_valid_096 = bool(
    audit_096["natural_channel_mask"]
    .astype(int)
    .eq(
        audit_096["source_natural_mask"].astype(int)
    ).all()
)

availability_valid_096 = bool(
    audit_096["available_channel_mask"]
    .astype(int)
    .eq(
        audit_096["expected_available_mask"].astype(int)
    ).all()
)

scheduled_subset_valid_096 = bool(
    (
        audit_096["scheduled_channel_mask"].astype(int)
        &
        ~audit_096["expected_available_mask"].astype(int)
    ).eq(0).all()
)

# Verify every active day has exactly one primary.
primary_valid_096 = bool(
    audit_096.loc[
        audit_096["audited_active_day"],
        "primary_channel_mask",
    ].astype(int).isin(CHANNEL_BITS_096).all()
    and
    (
        audit_096.loc[
            audit_096["audited_active_day"],
            "primary_channel_mask",
        ].astype(int)
        &
        audit_096.loc[
            audit_096["audited_active_day"],
            "scheduled_channel_mask",
        ].astype(int)
    ).eq(
        audit_096.loc[
            audit_096["audited_active_day"],
            "primary_channel_mask",
        ].astype(int)
    ).all()
)

# Forced off-days must contain no scheduled attack action.
offday_rows_096 = audit_096.loc[
    audit_096["audited_forced_offday"]
]


# Forced off-days must contain no scheduled attack action.
offday_rows_096 = audit_096.loc[
    audit_096["audited_forced_offday"]
]

offday_valid_096 = bool(
    offday_rows_096["scheduled_channel_mask"]
    .astype(int).eq(0).all()
    and
    offday_rows_096["primary_channel_mask"]
    .astype(int).eq(0).all()
    and
    ~offday_rows_096["is_switch"].astype(bool).any()
)


active_cardinality_valid_096 = bool(
    audit_096.loc[
        audit_096["audited_active_day"],
        "audited_channel_count",
    ].between(1, 3).all()
)

# Recompute synchronization labels rather than trust them.
sync_labels_valid_096 = bool(
    audit_096["sync_type"]
    .astype("string")
    .eq(
        audit_096["audited_sync_type"].astype("string")
    ).all()
)

# Recompute switches rather than trust stored flags.
switch_flags_valid_096 = bool(
    audit_096["is_switch"].astype(bool)
    .eq(
        audit_096["audited_switch"].astype(bool)
    ).all()
)

# Per-user target reconciliation.
user_counts_valid_096 = bool(
    independent_user_candidate_096["window_rows"]
    .eq(
        independent_user_candidate_096[
            "target_window_workdays"
        ]
    ).all()
    and
    independent_user_candidate_096["active_days"]
    .eq(
        independent_user_candidate_096[
            "target_active_days"
        ]
    ).all()
    and
    independent_user_candidate_096["forced_offdays"]
    .eq(
        independent_user_candidate_096[
            "target_forced_offdays"
        ]
    ).all()
)

sync_targets_valid_096 = bool(
    independent_user_candidate_096["two_channel_days"]
    .eq(
        independent_user_candidate_096[
            "target_2ch_days"
        ]
    ).all()
    and
    independent_user_candidate_096["three_channel_days"]
    .eq(
        independent_user_candidate_096[
            "target_3ch_days"
        ]
    ).all()
)

primary_coverage_valid_096 = bool(
    independent_user_candidate_096[
        "primary_coverage_mask"
    ].eq(
        independent_user_candidate_096[
            "channel_configuration"
        ].map(CONFIG_MASKS_096)
    ).all()
)

# User-day source eligibility is independently verified.
source_workdays_valid_096 = bool(
    audit_096["source_workday_eligible"]
    .eq(True).all()
)

# Off-day assignments are compared with the earlier
# reconstructed windows; they are not regenerated here.
window_status_096 = audit_096[
    ["user", "day", "is_forced_offday"]
].merge(
    windows_096[
        ["user", "day", "is_forced_offday"]
    ],
    on=["user", "day"],
    how="outer",
    suffixes=("_schedule", "_window"),
    indicator=True,
    validate="one_to_one",
)

forced_offday_assignment_valid_096 = bool(
    window_status_096["_merge"].eq("both").all()
    and
    window_status_096["is_forced_offday_schedule"]
    .eq(
        window_status_096[
            "is_forced_offday_window"
        ]
    ).all()
)

# Check the selected scenario identity separately.
scenario_identity_valid_096 = bool(
    same_values_096(
        independent_user_candidate_096[
            "target_scenario_option_id"
        ],
        independent_user_candidate_096[
            "scenario_option_id"
        ],
    )
)

# Approved natural channel capacity is checked directly
# using the original source events.
three_channel_valid_096 = bool(
    audit_096.loc[
        audit_096["audited_three_channel_day"],
        "expected_available_mask",
    ].astype(int).eq(7).all()
)


# ------------------------------------------------------------
# STEP 8 — Consolidated independent QA
# ------------------------------------------------------------

checks_096 = {
    "400 approved attacker identities":
        len(approved_users_096) == 400
        and scheduled_users_096 == approved_users_096,

    "Exact original daily window keys":
        window_keys_096 == schedule_keys_096,

    "Approved scenario attributes unchanged":
        immutability_valid_096,

    "Approved scenario identities unchanged":
        scenario_identity_valid_096,

    "Window boundaries and workday indices valid":
        date_bounds_valid_096 and workday_index_valid_096,

    "Original source workday eligibility":
        source_workdays_valid_096,

    "Natural channels match original event counts":
        natural_mask_valid_096,

    "Approved daily channel availability matches":
        availability_valid_096,

    "Scheduled channels are naturally available":
        scheduled_subset_valid_096,

    "Valid Primary Channel membership":
        primary_valid_096,

    "Forced off-day assignments and zero activity":
        forced_offday_assignment_valid_096
        and offday_valid_096,

    "Valid active channel cardinalities and labels":
        active_cardinality_valid_096
        and sync_labels_valid_096,

    "Active and forced off-day counts match targets":
        user_counts_valid_096,

    "Two- and three-channel targets realized":
        sync_targets_valid_096
        and three_channel_valid_096,

    "Primary Channel coverage complete":
        primary_coverage_valid_096,

    "Switching flags independently verified":
        switch_flags_valid_096,

    "Realized switches satisfy approved levels":
        independent_user_candidate_096[
            "switching_level_valid"
        ].all(),

    "Switch counts match feasibility evidence":
        independent_user_candidate_096[
            "attainable_switch_valid"
        ].all(),
}

independent_schedule_qa_09 = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if bool(ok) else "FAIL",
    }
    for name, ok in checks_096.items()
])

failed_096 = independent_schedule_qa_09.loc[
    independent_schedule_qa_09[
        "status"
    ].ne("PASS")
]

if not failed_096.empty:

    print("\nFAILED INDEPENDENT VALIDATION CHECKS")
    display(failed_096)

    raise RuntimeError(
        "Independent schedule validation failed. "
        "The schedule is not approved for export."
    )


# ------------------------------------------------------------
# STEP 9 — Publish validated audit results
# ------------------------------------------------------------

independent_schedule_user_audit_09 = (
    independent_user_candidate_096.copy(deep=True)
)

independent_schedule_duration_summary_09 = (
    independent_schedule_user_audit_09
    .groupby("window_workdays")
    .agg(
        attackers=("user", "size"),
        active_days=("active_days", "sum"),
        forced_offdays=("forced_offdays", "sum"),
        single_channel_days=("single_channel_days", "sum"),
        two_channel_days=("two_channel_days", "sum"),
        three_channel_days=("three_channel_days", "sum"),
        primary_switches=("switches", "sum"),
    )
    .reset_index()
)


# ------------------------------------------------------------
# STEP 10 — Readable results
# ------------------------------------------------------------

print("\nINDEPENDENTLY VERIFIED DAILY ACTIVITY")

display(
    independent_schedule_duration_summary_09.rename(
        columns={
            "window_workdays": "Duration",
            "attackers": "Attackers",
            "active_days": "Active days",
            "forced_offdays": "Forced off-days",
            "single_channel_days": "Single-channel",
            "two_channel_days": "Two-channel",
            "three_channel_days": "Three-channel",
            "primary_switches": "Switches",
        }
    )
)


# ------------------------------------------------------------
# STEP 11 — Concise validation summary
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("SECTION 9.6 — INDEPENDENT VALIDATION")
print("=" * 55)

print(
    "Approved attackers    :",
    len(independent_schedule_user_audit_09)
)

print(
    "Validated window days :",
    len(audit_096)
)

print(
    "Active days           :",
    int(audit_096["audited_active_day"].sum())
)

print(
    "Forced off-days       :",
    int(audit_096["audited_forced_offday"].sum())
)

print(
    "Two-channel days      :",
    int(audit_096["audited_two_channel_day"].sum())
)

print(
    "Three-channel days    :",
    int(audit_096["audited_three_channel_day"].sum())
)

print(
    "Primary switches      :",
    int(audit_096["audited_switch"].sum())
)

print(
    "Independent QA        :",
    f"{len(independent_schedule_qa_09)}/"
    f"{len(independent_schedule_qa_09)} PASS"
)

print("Validation status     : APPROVED IN MEMORY")
print("Attack events injected: NO")
print("Source files modified : NO")
print("Files exported        : NO")
print("=" * 55)
print("SECTION 9.6 — PASS")



INDEPENDENTLY VERIFIED DAILY ACTIVITY


,Duration,Attackers,Active days,Forced off-days,Single-channel,Two-channel,Three-channel,Switches
0,5,50,250,0,250,0,0,97
1,10,70,610,90,546,64,0,217
2,20,70,1215,185,1079,136,0,443
3,30,70,1775,325,1565,210,0,622
4,40,70,2335,465,2067,249,19,799
5,50,70,2895,605,2566,311,18,938



SECTION 9.6 — INDEPENDENT VALIDATION
Approved attackers    : 400
Validated window days : 10750
Active days           : 9080
Forced off-days       : 1670
Two-channel days      : 970
Three-channel days    : 37
Primary switches      : 3116
Independent QA        : 18/18 PASS
Validation status     : APPROVED IN MEMORY
Attack events injected: NO
Source files modified : NO
Files exported        : NO
SECTION 9.6 — PASS



## 9.7 — Final Dry-Run Export and Artifact Integrity

### Objective

Export the independently validated pre-injection attack schedule and its supporting evidence as a reproducible handoff for the subsequent injection stage.

### Export Preconditions

The export is permitted only when:

- Section 9.5.1 synchronization selection QA passes.
- Section 9.5.2 daily sequence construction QA passes.
- Section 9.6 independent schedule validation QA passes.
- All 400 approved attackers and their scenario identities are preserved.
- Every approved user-workday has exactly one schedule record.
- Realized synchronization and switching counts match the independently validated results.
- Forced off-days contain no scheduled attack actions.

### Final Artifacts

The export contains:

1. `daily_attack_schedule_r62.parquet` — Final daily channel schedule.
2. `daily_attack_schedule_r62.csv` — Readable copy of the daily schedule.
3. `selected_sync_targets_r62.parquet` — Selected synchronization targets.
4. `independent_schedule_user_audit.csv` — Per-attacker independent validation results.
5. `independent_schedule_duration_summary.csv` — Aggregate results by duration.
6. `synchronization_configuration_distribution.csv` — Selected synchronization patterns.
7. `synchronization_exceptions.csv` — Documented fallback assignments.
8. `final_validation_qa.csv` — Consolidated validation evidence.
9. `dry_run_data_dictionary.csv` — Exported field definitions and data types.
10. `notebook_summary.md` — Final methodology and result summary.
11. `export_manifest.json` — Artifact inventory, sizes, row counts, and SHA-256 hashes.

### Export Protection

- Create outputs in a temporary staging directory.
- Verify the exported Parquet and CSV files before publication.
- Compute SHA-256 hashes for all exported artifacts.
- Publish the output directory only after all validation checks pass.
- Refuse to overwrite an existing final export automatically.
- Preserve all original source datasets and previously approved notebook outputs.

### Handoff Boundary

The exported schedule describes planned attack-channel activity.

It does not contain injected attack events, modified event counts, or completed ground-truth injection labels.

Actual event-count injection and its subsequent validation belong to the next stage.

### Acceptance Criteria

The final export is approved only when every required artifact is present, readable, and integrity-verified.

### Expected Outcome

A complete, independently validated, checksum-protected pre-injection schedule ready for the next notebook.


In [20]:

# ============================================================
# NOTEBOOK 09 — SECTION 9.7
# FINAL DRY-RUN EXPORT AND ARTIFACT INTEGRITY
#
# Validated handoff | SHA-256 | Protected publication
# No injection | No source modification
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import tempfile

import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# STEP 1 — Final export prerequisites
# ------------------------------------------------------------

QA_SOURCES_097 = {
    "9.5.1 Synchronization Selection":
        synchronization_selection_qa_09,
    "9.5.2 Daily Sequence Construction":
        daily_sequence_qa_09,
    "9.6 Independent Validation":
        independent_schedule_qa_09,
}

failed_prerequisites_097 = {
    name: qa.loc[qa["status"].ne("PASS")]
    for name, qa in QA_SOURCES_097.items()
    if (
        qa.empty
        or qa["status"].isna().any()
        or not qa["status"].eq("PASS").all()
    )
}

if failed_prerequisites_097:
    raise RuntimeError(
        "Previous notebook QA is incomplete or failed: "
        + ", ".join(failed_prerequisites_097)
    )

schedule_097 = daily_attack_schedule_09.copy(deep=True)
targets_097 = selected_sync_targets_09.copy(deep=True)
user_audit_097 = (
    independent_schedule_user_audit_09.copy(deep=True)
)
duration_audit_097 = (
    independent_schedule_duration_summary_09.copy(deep=True)
)
configuration_097 = (
    synchronization_configuration_table_09.copy(deep=True)
)
exceptions_097 = (
    synchronization_exceptions_09.copy(deep=True)
)
approved_097 = approved_assignments_09.copy(deep=True)

for frame in (
    schedule_097,
    targets_097,
    user_audit_097,
    approved_097,
):
    frame["user"] = frame["user"].astype("string")


# ------------------------------------------------------------
# STEP 2 — Reconcile approved handoff
# ------------------------------------------------------------

def identity_pairs_097(frame):
    return set(
        zip(
            frame["user"].astype(str),
            frame["scenario_option_id"].astype("string"),
        )
    )


approved_pairs_097 = identity_pairs_097(approved_097)

identity_valid_097 = (
    len(approved_097) == 400
    and approved_097["user"].is_unique
    and len(targets_097) == 400
    and targets_097["user"].is_unique
    and len(user_audit_097) == 400
    and user_audit_097["user"].is_unique
    and identity_pairs_097(targets_097) == approved_pairs_097
    and identity_pairs_097(user_audit_097) == approved_pairs_097
)

schedule_097["day"] = pd.to_datetime(
    schedule_097["day"], errors="raise"
)

schedule_keys_valid_097 = bool(
    len(schedule_097) == 10750
    and not schedule_097.duplicated(
        ["user", "day"]
    ).any()
    and set(schedule_097["user"].astype(str))
    == set(approved_097["user"].astype(str))
)

active_097 = schedule_097["is_active_day"].astype(bool)
offday_097 = schedule_097["is_forced_offday"].astype(bool)

daily_status_valid_097 = bool(
    active_097.eq(~offday_097).all()
    and int(active_097.sum()) == 9080
    and int(offday_097.sum()) == 1670
)

offday_actions_valid_097 = bool(
    schedule_097.loc[
        offday_097, "scheduled_channel_mask"
    ].astype(int).eq(0).all()
    and
    schedule_097.loc[
        offday_097, "primary_channel_mask"
    ].astype(int).eq(0).all()
    and
    ~schedule_097.loc[
        offday_097, "is_switch"
    ].astype(bool).any()
)

two_days_097 = int(
    schedule_097["scheduled_channel_count"]
    .eq(2).sum()
)

three_days_097 = int(
    schedule_097["scheduled_channel_count"]
    .eq(3).sum()
)

switches_097 = int(
    schedule_097["is_switch"].astype(bool).sum()
)

totals_valid_097 = bool(
    two_days_097
    == int(targets_097["target_2ch_days"].sum())
    == int(user_audit_097["two_channel_days"].sum())
    and
    three_days_097
    == int(targets_097["target_3ch_days"].sum())
    == int(user_audit_097["three_channel_days"].sum())
    and
    switches_097
    == int(user_audit_097["switches"].sum())
)

duration_valid_097 = bool(
    user_audit_097["window_workdays"]
    .value_counts().to_dict()
    == {
        int(k): int(v)
        for k, v in DURATION_TARGETS_09.items()
    }
)

preexport_checks_097 = {
    "Approved attacker and scenario identities":
        identity_valid_097,
    "Complete unique daily schedule":
        schedule_keys_valid_097,
    "Active and forced off-day totals":
        daily_status_valid_097,
    "No attack actions on forced off-days":
        offday_actions_valid_097,
    "Synchronization and switching totals":
        totals_valid_097,
    "Approved duration distribution":
        duration_valid_097,
}

failed_preexport_097 = [
    name
    for name, passed in preexport_checks_097.items()
    if not bool(passed)
]

if failed_preexport_097:
    raise RuntimeError(
        "Final export blocked: "
        + "; ".join(failed_preexport_097)
    )


# ------------------------------------------------------------
# STEP 3 — Assemble consolidated QA
# ------------------------------------------------------------

final_validation_qa_097 = pd.concat(
    [
        qa.assign(section=section)
        for section, qa in QA_SOURCES_097.items()
    ],
    ignore_index=True,
)

preexport_qa_097 = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "section": "9.7 Export Preconditions",
    }
    for name, passed in preexport_checks_097.items()
])

final_validation_qa_097 = pd.concat(
    [
        final_validation_qa_097,
        preexport_qa_097,
    ],
    ignore_index=True,
)[["section", "check", "status"]]

if not final_validation_qa_097["status"].eq("PASS").all():
    raise RuntimeError("Consolidated validation failed.")


# ------------------------------------------------------------
# STEP 4 — Prepare data dictionary
# ------------------------------------------------------------

FIELD_DESCRIPTIONS_097 = {
    "user": "User identifier",
    "scenario_option_id": "Approved scenario identifier",
    "day": "Calendar date of the scheduled workday",
    "workday_index": "Position within the approved workday window",
    "window_workdays": "Approved attack duration in workdays",
    "window_start": "Approved attack window start",
    "window_end": "Approved attack window end",
    "channel_configuration": "Approved attack channel configuration",
    "switching_level": "Approved primary-channel switching level",
    "stealth_level": "Approved stealth category",
    "forced_offdays": "Approved number of forced off-days",
    "is_forced_offday": "Whether injection is prohibited on this day",
    "is_active_day": "Whether this is an attack-active workday",
    "available_channel_mask":
        "Naturally active approved channels; E=1, U=2, H=4",
    "available_channel_set":
        "Readable naturally available approved channels",
    "natural_channel_mask":
        "Naturally active channels from source user-day counts",
    "primary_channel_mask":
        "Selected primary channel; E=1, U=2, H=4; 0=none",
    "primary_channel":
        "Readable selected primary channel",
    "scheduled_channel_mask":
        "Planned channel set using E=1, U=2, H=4",
    "scheduled_channel_set":
        "Readable planned channel set",
    "scheduled_channel_count":
        "Number of distinct planned channels on this day",
    "sync_type":
        "FORCED_OFFDAY, SINGLE, TWO_CHANNEL, or THREE_CHANNEL",
    "is_switch":
        "Primary channel changed since previous active day",
    "active_day_index":
        "Sequential index across active days only; 0 on off-days",
    "target_2ch_days":
        "Selected exactly-two-channel synchronization days",
    "target_3ch_days":
        "Selected three-channel synchronization days",
    "attainable_switch_mask":
        "Bitmask of jointly feasible approved switch counts",
    "rule_status":
        "Full rule, two-channel fallback, or zero assignment",
    "fallback_reason":
        "Reason for permitted synchronization fallback",
    "selection_method":
        "Deterministic synchronization selection method",
    "selection_seed":
        "Seed used for reproducible synchronization selection",
}

exports_097 = {
    "daily_attack_schedule_r62.parquet":
        schedule_097,
    "daily_attack_schedule_r62.csv":
        schedule_097,
    "selected_sync_targets_r62.parquet":
        targets_097,
    "independent_schedule_user_audit.csv":
        user_audit_097,
    "independent_schedule_duration_summary.csv":
        duration_audit_097,
    "synchronization_configuration_distribution.csv":
        configuration_097,
    "synchronization_exceptions.csv":
        exceptions_097,
    "final_validation_qa.csv":
        final_validation_qa_097,
}

dictionary_rows_097 = []

for filename, frame in exports_097.items():
    for column in frame.columns:
        dictionary_rows_097.append({
            "file": filename,
            "field": column,
            "dtype": str(frame[column].dtype),
            "description": FIELD_DESCRIPTIONS_097.get(
                column,
                "Supporting source, scenario, or validation field",
            ),
        })

dry_run_data_dictionary_097 = pd.DataFrame(
    dictionary_rows_097
)


# ------------------------------------------------------------
# STEP 5 — Build final notebook summary
# ------------------------------------------------------------

qa_passes_097 = len(final_validation_qa_097)

summary_text_097 = f"""# Notebook 09 — Pre-Injection Dry Run

## Purpose

Prepare and independently validate an attack-channel schedule
for the 400 approved attacker scenarios.

## Inputs

- Approved Notebook 08 scenario assignments
- Original processed user-day activity
- Validated daily channel availability
- Selected synchronization targets
- Independently validated daily channel schedules

## Approved Results

| Metric | Value |
|---|---:|
| Approved attackers | 400 |
| Window user-workdays | {len(schedule_097):,} |
| Active user-workdays | {int(active_097.sum()):,} |
| Forced off-days | {int(offday_097.sum()):,} |
| Exactly-two-channel days | {two_days_097:,} |
| Three-channel days | {three_days_097:,} |
| Primary channel switches | {switches_097:,} |
| Consolidated QA checks | {qa_passes_097}/{qa_passes_097} PASS |

## Methodology

1. Verify approved inputs and scenario assignments.
2. Reconstruct daily attack windows and natural availability.
3. Analyze natural synchronization opportunities.
4. Establish joint switching-synchronization feasibility.
5. Select duration-rule-based synchronization targets.
6. Reconstruct feasible daily Primary Channel sequences.
7. Independently validate schedules against original user-day
   counts and approved scenario constraints.
8. Export validated plans and integrity evidence.

## Synchronization Selection

The selected targets follow Table B.3 with approved
two-channel-only and zero-synchronization fallbacks.

Alternative jointly feasible configurations are selected
deterministically without always preferring the minimum.

## Switching

A switch occurs when the Primary Channel changes between
consecutive active days.

Forced off-days are excluded from switching transitions.

## Scope and Limitations

This notebook produces a planned attack schedule only.

It does not inject attack events, alter event counts,
modify original user-day data, or generate final injected
ground-truth labels.

The subsequent injection stage must independently validate
the planned actions against its generated event-count changes.

## Integrity

Every exported artifact is listed in export_manifest.json
with its SHA-256 checksum.

The manifest is not self-hashed.

Original source datasets and Notebook 08 outputs remain
unchanged.
"""


# ------------------------------------------------------------
# STEP 6 — Prepare protected export destination
# ------------------------------------------------------------

FINAL_REPORT_DIR_097 = Path(REPORT_09).resolve()

FINAL_REPORT_DIR_097.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if FINAL_REPORT_DIR_097.exists():
    raise FileExistsError(
        "Protected export already exists. "
        "Existing files were not replaced:\n"
        f"{FINAL_REPORT_DIR_097}"
    )

staging_dir_097 = Path(
    tempfile.mkdtemp(
        prefix=".09_dry_run_staging_",
        dir=FINAL_REPORT_DIR_097.parent,
    )
)


# ------------------------------------------------------------
# STEP 7 — Export, verify, hash, then publish
# ------------------------------------------------------------

def file_sha256_097(path):

    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        for chunk in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


artifact_records_097 = []

try:

    # Export tabular artifacts.
    for filename, frame in exports_097.items():

        output_path = staging_dir_097 / filename

        if filename.endswith(".parquet"):

            frame.to_parquet(
                output_path,
                index=False,
                engine="pyarrow",
            )

            # Validate full Parquet round-trip.
            restored = pd.read_parquet(output_path)

            
            # Verify Parquet round-trip without categorical dtype conflicts.
            # Original and restored values must still match exactly.

            original_check_097 = frame.reset_index(drop=True).copy()
            restored_check_097 = restored.reset_index(drop=True).copy()

            for column in original_check_097.columns:

                original_dtype = original_check_097[column].dtype
                restored_dtype = restored_check_097[column].dtype

                if (
                    isinstance(original_dtype, pd.CategoricalDtype)
                    or isinstance(restored_dtype, pd.CategoricalDtype)
                ):
                    original_check_097[column] = (
                        original_check_097[column].astype(object)
                    )

                    restored_check_097[column] = (
                        restored_check_097[column].astype(object)
                    )

            pd.testing.assert_frame_equal(
                original_check_097,
                restored_check_097,
                check_dtype=False,
                check_exact=True,
                check_categorical=False,
            )


        else:

            frame.to_csv(
                output_path,
                index=False,
                encoding="utf-8-sig",
            )

            # Verify CSV readability, rows, and headers.
            restored = pd.read_csv(
                output_path,
                encoding="utf-8-sig",
            )

            if (
                len(restored) != len(frame)
                or restored.columns.tolist()
                != frame.columns.tolist()
            ):
                raise RuntimeError(
                    f"CSV round-trip failed: {filename}"
                )

        artifact_records_097.append({
            "filename": filename,
            "format": output_path.suffix.lstrip("."),
            "rows": int(len(frame)),
            "columns": int(len(frame.columns)),
            "size_bytes": output_path.stat().st_size,
            "sha256": file_sha256_097(output_path),
        })

    # Data dictionary.
    dictionary_path_097 = (
        staging_dir_097 / "dry_run_data_dictionary.csv"
    )

    dry_run_data_dictionary_097.to_csv(
        dictionary_path_097,
        index=False,
        encoding="utf-8-sig",
    )

    dictionary_check_097 = pd.read_csv(
        dictionary_path_097,
        encoding="utf-8-sig",
    )

    if (
        len(dictionary_check_097)
        != len(dry_run_data_dictionary_097)
        or dictionary_check_097.columns.tolist()
        != dry_run_data_dictionary_097.columns.tolist()
    ):
        raise RuntimeError("Dictionary round-trip failed.")

    artifact_records_097.append({
        "filename": dictionary_path_097.name,
        "format": "csv",
        "rows": len(dry_run_data_dictionary_097),
        "columns": len(dry_run_data_dictionary_097.columns),
        "size_bytes": dictionary_path_097.stat().st_size,
        "sha256": file_sha256_097(dictionary_path_097),
    })

    # Notebook summary.
    summary_path_097 = (
        staging_dir_097 / "notebook_summary.md"
    )

    summary_path_097.write_text(
        summary_text_097,
        encoding="utf-8",
    )

    if summary_path_097.read_text(
        encoding="utf-8"
    ) != summary_text_097:
        raise RuntimeError("Notebook summary verification failed.")

    artifact_records_097.append({
        "filename": summary_path_097.name,
        "format": "md",
        "rows": None,
        "columns": None,
        "size_bytes": summary_path_097.stat().st_size,
        "sha256": file_sha256_097(summary_path_097),
    })

    # Manifest lists all non-manifest artifacts.
    manifest_097 = {
        "notebook": "09_PreInjection_Dry_Run",
        "dataset_version": "r6.2",
        "export_type": "PRE_INJECTION_DRY_RUN",
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "validation_status": "PASS",
        "approved_attackers": 400,
        "window_user_days": len(schedule_097),
        "active_user_days": int(active_097.sum()),
        "forced_offdays": int(offday_097.sum()),
        "two_channel_days": two_days_097,
        "three_channel_days": three_days_097,
        "primary_switches": switches_097,
        "qa_passes": qa_passes_097,
        "attack_events_injected": False,
        "source_files_modified": False,
        "manifest_self_hashed": False,
        "artifacts": artifact_records_097,
    }

    manifest_path_097 = (
        staging_dir_097 / "export_manifest.json"
    )

    manifest_path_097.write_text(
        json.dumps(
            manifest_097,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    # Verify every artifact against the manifest.
    loaded_manifest_097 = json.loads(
        manifest_path_097.read_text(encoding="utf-8")
    )

    for record in loaded_manifest_097["artifacts"]:

        path = staging_dir_097 / record["filename"]

        if not path.is_file():
            raise RuntimeError(
                f"Missing export artifact: {path.name}"
            )

        if path.stat().st_size != record["size_bytes"]:
            raise RuntimeError(
                f"Artifact size mismatch: {path.name}"
            )

        if file_sha256_097(path) != record["sha256"]:
            raise RuntimeError(
                f"SHA-256 mismatch: {path.name}"
            )

    expected_names_097 = {
        record["filename"]
        for record in artifact_records_097
    } | {"export_manifest.json"}

    actual_names_097 = {
        item.name for item in staging_dir_097.iterdir()
    }

    if actual_names_097 != expected_names_097:
        raise RuntimeError(
            "Export inventory does not match manifest."
        )

    # Publish only the fully verified directory.
    if FINAL_REPORT_DIR_097.exists():
        raise FileExistsError(
            "Final export destination appeared during "
            "staging. No replacement permitted."
        )

    os.rename(
        staging_dir_097,
        FINAL_REPORT_DIR_097,
    )

except Exception:

    if staging_dir_097.exists():
        shutil.rmtree(staging_dir_097)

    raise


# ------------------------------------------------------------
# STEP 8 — Final export report
# ------------------------------------------------------------

export_inventory_09 = pd.DataFrame(
    loaded_manifest_097["artifacts"]
)

export_manifest_09 = loaded_manifest_097

export_qa_09 = pd.DataFrame([
    {
        "check": "All prerequisite QA passed",
        "status": "PASS",
    },
    {
        "check": "All exported files verified",
        "status": "PASS",
    },
    {
        "check": "All SHA-256 hashes verified",
        "status": "PASS",
    },
    {
        "check": "Manifest inventory complete",
        "status": "PASS",
    },
    {
        "check": "Protected publication successful",
        "status": "PASS",
    },
])

print("\nFINAL EXPORTED ARTIFACTS")

display(
    export_inventory_09[
        ["filename", "rows", "size_bytes"]
    ].reset_index(drop=True)
)

print("\n" + "=" * 55)
print("SECTION 9.7 — FINAL EXPORT SUMMARY")
print("=" * 55)
print("Approved attackers     : 400")
print("Window user-days       :", len(schedule_097))
print("Active user-days       :", int(active_097.sum()))
print("Forced off-days        :", int(offday_097.sum()))
print("Two-channel days       :", two_days_097)
print("Three-channel days     :", three_days_097)
print("Primary switches       :", switches_097)
print("Exported files         :", len(expected_names_097))
print(
    "Export QA              :",
    f"{len(export_qa_09)}/{len(export_qa_09)} PASS"
)
print("SHA-256 verification   : PASS")
print("Attack events injected : NO")
print("Source files modified  : NO")
print("Export status          : COMPLETE")
print("Output directory       :", FINAL_REPORT_DIR_097)
print("=" * 55)
print("SECTION 9.7 — PASS")



FINAL EXPORTED ARTIFACTS


,filename,rows,size_bytes
0,daily_attack_schedule_r62.parquet,10750.0,130023
1,daily_attack_schedule_r62.csv,10750.0,2353528
2,selected_sync_targets_r62.parquet,400.0,26796
3,independent_schedule_user_audit.csv,400.0,52579
4,independent_schedule_duration_summary.csv,6.0,306
5,synchronization_configuration_distribution.csv,15.0,422
6,synchronization_exceptions.csv,115.0,22924
7,final_validation_qa.csv,47.0,3413
8,dry_run_data_dictionary.csv,188.0,19440
9,notebook_summary.md,NaN,2221



SECTION 9.7 — FINAL EXPORT SUMMARY
Approved attackers     : 400
Window user-days       : 10750
Active user-days       : 9080
Forced off-days        : 1670
Two-channel days       : 970
Three-channel days     : 37
Primary switches       : 3116
Exported files         : 11
Export QA              : 5/5 PASS
SHA-256 verification   : PASS
Attack events injected : NO
Source files modified  : NO
Export status          : COMPLETE
Output directory       : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\09_PreInjection_Dry_Run
SECTION 9.7 — PASS
